In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## v12 — Yangon–Pyay parcel-fee calculator
Adds `calculate_parcel_fee` for small consignments on Yangon–Pyay/Pyay–Yangon.
Rule: 13 pyas (0.13 MMK) per viss per mile; minimum 5 viss; minimum 100 miles;
final fee rounded upward to the applicable 50-MMK boundary. The tool reuses
`GET /api/routes/{route_id}` and `distance_from_origin`; no backend schema change is required.

## v13 — Stable catalog v2 support

The stable `railway_catalog.json` may now include these route-station fields:

- `route_station_id`: internal `route_stations.id`, used by the fare API
- `station_id`: public `stations.id`, used by schedules/parcel/station APIs
- `order`: station order on the route
- `distance_from_origin`: miles from that route's origin

Internal notation is now:

- `S` = public station ID
- `RS` = RouteStation row ID
- `R` = route ID
- `T` = train ID
- `@` = station order
- `mi` = miles from route origin

The parser remains backward-compatible with the old catalog. If `RS` or
`distance_from_origin` is absent, schedule/station tools still work and the
fare flow can fall back to `get_route`.

## v14 — Cloud Run backend configuration

The railway backend has moved from Render to Google Cloud Run.

Canonical backend:

`https://smart-railway-platform-git-803829337765.asia-southeast1.run.app`

Changes:
- the Cloud Run URL is now the notebook's default backend;
- a stale Colab Secret/environment value pointing to the old Render host is
  detected and ignored;
- `RAILWAY_API_BASE_URL` may still override the default for future deployments;
- browser CORS is kept separate from Colab/server-to-server API access.

## v15 — API and KB fixes

Runtime-log fixes:
- fare API uses `ECONOMY_CLASS`, `UPPER_CLASS`, `SLEEPER`;
- legacy `ORDINARY` is normalized to `ECONOMY_CLASS`;
- passenger-rich RAG collection defaults to `documents_v4_1`;
- `RAILWAY_KB_COLLECTION` may override that collection;
- knowledge questions should remain on RAG instead of unrelated railway APIs;
- MongoDB collection/document-count diagnostics are printed at startup;
- large seat maps are compacted before being sent back to the LLM.

## v16 — Train service type vs rolling-stock aliases

The chatbot now distinguishes:

- **service type**: backend `train_type`, e.g. `EXPRESS`, `LOCAL`
- **rolling-stock type**: passenger wording such as `DEMU`

A train may therefore be both:

```text
service_type = EXPRESS
rolling_stock_type = DEMU
```

without changing the backend enum.

For the Yangon–Pyay demo, stable passenger aliases are added for:

- Train 73 → DEMU
- Train 74 → DEMU

The catalog parser also supports optional future JSON fields:

- `rolling_stock_type`
- `aliases`
- `service_tags`

**v11 backend operational-data integration**
- Keeps the v9 Gemini function-calling / thought-signature fix.
- Adds backend-grounded fare calculation and fare-matrix tools.
- Adds schedule-specific coach/seat availability tools.
- Adds read-only ticket journey/booking-status lookup.
- Booking writes (`reserve`, `confirm`, `cancel`) are intentionally not exposed to the LLM.
- Fare calls use **RouteStation IDs** from the selected route; public Station IDs are never substituted.
- Dynamic operational answers continue to fail closed instead of guessing.


# RailMM — Gemini Primary + Qwen Fallback + Backend Operational Tools

**Architecture**
- BGE-M3 multilingual retrieval over MongoDB Atlas for stable/reference railway knowledge.
- **Auto mode:** Vertex AI Gemini primary, Groq Qwen fallback when the Gemini request fails.
- Gradio can also lock a turn to Gemini or Qwen.
- Stable station/train/route identifiers come from `railway_catalog.json`.
- FastAPI remains authoritative for changing operational facts.

**Backend-grounded passenger tools**
- Station/train/route lookup and train-stop lookup.
- Dated A → B schedule search with station-specific times.
- Exact configured fare calculation and fare matrices.
- Schedule-specific coach/seat availability and exact-seat checks.
- Read-only ticket journey/booking status.

**Safety boundary**
- RailMM does not reserve, confirm, cancel, refund, or mutate bookings.
- The chatbot does not fetch the full booking-by-ticket record because it may contain passenger identity/contact data.
- It never invents fares, seats, times, booking status, or internal IDs.

**Colab settings/secrets**
`MONGO_URI`, `GROQ_API_KEY`, plus the existing Vertex/Gemini configuration used by this notebook. `RAILWAY_API_BASE_URL` is now optional because the Cloud Run production URL is the default.


In [ ]:
# import sys

# # 1. Add the folder path to the system path
# sys.path.append('/content/drive/MyDrive/Colab Notebooks/RailwayBot')

# # 2. Import the classes normally
# from embedding_service import EmbeddingService
# from mongo_service import MongoService

In [ ]:
!pip -q install \
sentence-transformers \
pymongo \
fastapi \
uvicorn \
nest_asyncio \
httpx \
python-dotenv \
requests \
groq \
google-genai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 10.1 MB/s eta 0:00:00


In [ ]:
!pip install -q gradio

In [ ]:
# Optional: used only when direct Myanmar retrieval is weak.
!pip install -q deep-translator


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.3 MB/s eta 0:00:00


In [ ]:
from dataclasses import dataclass
import os
from getpass import getpass


def _read_colab_secret(name: str):
    """Read a Colab Secret when available, otherwise return None."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


def _get_required_setting(name: str, prompt: str, secret: bool = False) -> str:
    value = _read_colab_secret(name) or os.getenv(name)
    if value:
        return value.strip()

    value = getpass(prompt) if secret else input(prompt)
    value = value.strip()

    if not value:
        raise ValueError(f"{name} is required.")

    return value


@dataclass
class Config:
    # ============================================================
    # MongoDB Atlas
    # ============================================================
    MONGO_URI = _get_required_setting(
        "MONGO_URI",
        "Enter MongoDB Atlas URI: ",
        secret=True
    )
    DATABASE_NAME = "railway_chatbot"

    # Passenger-rich MongoDB KB.
    COLLECTION_NAME = (
        _read_colab_secret("RAILWAY_KB_COLLECTION")
        or os.getenv("RAILWAY_KB_COLLECTION")
        or "documents_v4_1"
    ).strip()

    VECTOR_INDEX = "vector_index"

    # ============================================================
    # Embeddings
    # BGE-M3 is multilingual, so Myanmar queries are searched
    # directly first. No translation is needed in the normal path.
    # ============================================================
    EMBEDDING_MODEL = "BAAI/bge-m3"
    EMBEDDING_DEVICE = "cpu"

    # ============================================================
    # LLM PROVIDERS
    # Auto mode = Vertex AI Gemini primary -> Groq Qwen fallback.
    # Gemini/Qwen modes lock to the selected provider.
    # ============================================================
    GOOGLE_CLOUD_PROJECT_ID = (
        _read_colab_secret("GOOGLE_CLOUD_PROJECT_ID")
        or os.getenv("GOOGLE_CLOUD_PROJECT_ID")
        or ""
    )
    GEMINI_MODEL = os.getenv(
        "GEMINI_MODEL",
        "gemini-3.5-flash"
    )
    GEMINI_LOCATION = os.getenv(
        "GEMINI_LOCATION",
        "global"
    )

    GROQ_API_KEY = _get_required_setting(
        "GROQ_API_KEY",
        "Enter Groq API key: ",
        secret=True
    )
    QWEN_MODEL = "qwen/qwen3.6-27b"
    QWEN_REASONING_EFFORT = "none"

    # Legacy alias kept so older diagnostic cells do not break.
    LLM_MODEL = QWEN_MODEL
    LLM_TIMEOUT = 90
    DEFAULT_MODEL_MODE = "Auto"

    # ============================================================
    # Translation
    # Translation is NOT used for normal generation anymore.
    # It is kept only as a retrieval fallback:
    #
    # Myanmar query -> BGE-M3 direct search
    # If score is weak -> translate query to English -> search again
    #
    # Final answers are always generated directly by Qwen.
    # ============================================================
    TRANSLATION_FALLBACK_ENABLED = True
    TRANSLATION_FALLBACK_SCORE = 0.70

    # ============================================================
    # RAG
    # ============================================================
    TOP_K = 8
    MIN_SCORE = 0.65

    # ============================================================
    # Prompt budget (v8 Pyay-focused KB)
    # Keep the Groq/Qwen request comfortably below the free-tier TPM cap.
    # ============================================================
    PROMPT_HISTORY_MESSAGES = 3
    PROMPT_HISTORY_CHARS_PER_MESSAGE = 350
    PROMPT_DOC_LIMIT = 3
    PROMPT_DOC_CHARS = 700

    # ============================================================
    # Cache
    # ============================================================
    ENABLE_CACHE = True
    CACHE_TTL = 3600

    # ============================================================
    # LLM Generation
    # ============================================================
    LLM_TEMPERATURE = 0.3
    LLM_MAX_TOKENS = 512
    LLM_TOP_P = 0.9

    # ============================================================
    # Deployed Railway Backend — Google Cloud Run
    # ============================================================
    #
    # Production URL is intentionally NOT treated as a secret.
    # Colab -> FastAPI is server-to-server traffic, so browser CORS
    # restrictions do not apply to these requests.
    #
    # An optional RAILWAY_API_BASE_URL secret/environment variable can
    # override this URL for a future deployment. A stale value pointing
    # to the retired Render service is ignored automatically.
    # ============================================================
    RAILWAY_CLOUD_RUN_URL = (
        "https://smart-railway-platform-git-803829337765.asia-southeast1.run.app"
    )

    _configured_railway_api_url = (
        _read_colab_secret("RAILWAY_API_BASE_URL")
        or os.getenv("RAILWAY_API_BASE_URL")
        or RAILWAY_CLOUD_RUN_URL
    ).strip().rstrip("/")

    if "smart-railway-api.onrender.com" in _configured_railway_api_url.casefold():
        print(
            "⚠️ Ignoring retired Render RAILWAY_API_BASE_URL; "
            "using Google Cloud Run instead."
        )
        _configured_railway_api_url = RAILWAY_CLOUD_RUN_URL

    RAILWAY_API_BASE_URL = _configured_railway_api_url
    RAILWAY_API_TIMEOUT = 20

    # ============================================================
    # Stable railway reference catalog
    # Generate this JSON from Neon using railway_catalog_export.sql.
    # After DB reference-data changes, replace this file and call
    # refresh_railway_catalog(); the notebook itself does not change.
    # ============================================================
    RAILWAY_CATALOG_PATH = os.getenv(
        "RAILWAY_CATALOG_PATH",
        "/content/drive/MyDrive/ChabotData/railway_catalog.json"
    )


print("✅ Config loaded")
print("Railway backend:", Config.RAILWAY_API_BASE_URL)
print("Model mode:", Config.DEFAULT_MODEL_MODE)
print("Primary LLM:", Config.GEMINI_MODEL, "(Vertex AI)")
print("Fallback LLM:", Config.QWEN_MODEL, "(Groq)")
print("Embedding:", Config.EMBEDDING_MODEL, f"({Config.EMBEDDING_DEVICE})")
print("Knowledge collection:", Config.COLLECTION_NAME)
print("Translation: fallback only")
print("Railway catalog:", Config.RAILWAY_CATALOG_PATH)


✅ Config loaded
Railway backend: https://smart-railway-platform-git-803829337765.asia-southeast1.run.app
Model mode: Auto
Primary LLM: gemini-3.5-flash (Vertex AI)
Fallback LLM: qwen/qwen3.6-27b (Groq)
Embedding: BAAI/bge-m3 (cpu)
Knowledge collection: documents_v4_1
Translation: fallback only
Railway catalog: /content/drive/MyDrive/ChabotData/railway_catalog.json


In [ ]:
# ============================================================
# v14 CLOUD RUN BACKEND CONFIGURATION SMOKE TEST
# ============================================================

_expected_cloud_run = (
    "https://smart-railway-platform-git-803829337765.asia-southeast1.run.app"
)

assert Config.RAILWAY_API_BASE_URL.startswith("https://")
assert "smart-railway-api.onrender.com" not in (
    Config.RAILWAY_API_BASE_URL.casefold()
)

print("✅ v14 backend configuration passed")
print("Resolved backend:", Config.RAILWAY_API_BASE_URL)

if Config.RAILWAY_API_BASE_URL != _expected_cloud_run:
    print(
        "ℹ️ RAILWAY_API_BASE_URL is using an explicit override. "
        "That is fine if the override is intentional."
    )


✅ v14 backend configuration passed
Resolved backend: https://smart-railway-platform-git-803829337765.asia-southeast1.run.app


In [ ]:
# ============================================================
# STABLE RAILWAY REFERENCE — LOAD ONCE, KEEP AS SYSTEM KNOWLEDGE
# ============================================================
# The API LLM itself is stateless; it cannot permanently memorize these IDs.
# This cell provides the practical equivalent:
#   1) load the stable reference ONCE when the notebook starts,
#   2) normalize it,
#   3) compress it into ~small system knowledge,
#   4) reuse the same in-memory reference for Gemini/Qwen calls.
#
# The JSON file is NOT read on every passenger question.
# Schedules/live information are intentionally NOT included here.

from pathlib import Path
from typing import Dict, Any
import json
import re
import unicodedata

_MYANMAR_DIGITS = "၀၁၂၃၄၅၆၇၈၉"
_ASCII_DIGITS = "0123456789"
_MY_TO_ASCII = str.maketrans(_MYANMAR_DIGITS, _ASCII_DIGITS)


def _catalog_norm(value: Any) -> str:
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = text.translate(_MY_TO_ASCII).casefold().strip()
    return re.sub(r"[\s\-\(\)\[\],._/]+", "", text)


# ============================================================
# PASSENGER-FACING TRAIN DESCRIPTORS
# ============================================================
#
# `train_type` is the backend SERVICE classification.
# DEMU is rolling-stock terminology.
#
# Prefer values exported in railway_catalog.json when present.
# These are stable Yangon–Pyay fallback aliases.
#
_DEFAULT_TRAIN_PUBLIC_DESCRIPTORS = {
    "73": {
        "rolling_stock_type": "DEMU",
        "aliases": [
            "DEMU",
            "Diesel Electric Multiple Unit",
            "diesel-electric multiple unit",
            "ဒီအီးအမ်ယူ",
            "ဒီအမ်ယူ",
        ],
        "service_tags": [
            "Yangon-Pyay DEMU",
        ],
    },
    "74": {
        "rolling_stock_type": "DEMU",
        "aliases": [
            "DEMU",
            "Diesel Electric Multiple Unit",
            "diesel-electric multiple unit",
            "ဒီအီးအမ်ယူ",
            "ဒီအမ်ယူ",
        ],
        "service_tags": [
            "Pyay-Yangon DEMU",
        ],
    },
}


def _clean_string_list(value):

    if value is None:
        return []

    if isinstance(value, str):
        items = [value]
    elif isinstance(value, (list, tuple, set)):
        items = list(value)
    else:
        items = [str(value)]

    result = []

    for item in items:
        text = str(item or "").strip()

        if text and text not in result:
            result.append(text)

    return result


def _train_public_descriptor(train: dict) -> dict:
    """
    Merge optional catalog descriptors with stable fallback metadata.

    Catalog values take precedence over fallback values.
    """
    train_no = str(
        train.get("train_no") or ""
    ).strip()

    fallback = (
        _DEFAULT_TRAIN_PUBLIC_DESCRIPTORS.get(
            train_no,
            {}
        )
    )

    rolling_stock = (
        str(
            train.get("rolling_stock_type")
            or ""
        ).strip()
        or str(
            fallback.get("rolling_stock_type")
            or ""
        ).strip()
    )

    aliases = _clean_string_list(
        train.get("aliases")
    )

    for alias in _clean_string_list(
        fallback.get("aliases")
    ):
        if alias not in aliases:
            aliases.append(alias)

    service_tags = _clean_string_list(
        train.get("service_tags")
    )

    for tag in _clean_string_list(
        fallback.get("service_tags")
    ):
        if tag not in service_tags:
            service_tags.append(tag)

    return {
        "rolling_stock_type": rolling_stock,
        "aliases": aliases,
        "service_tags": service_tags,
    }


class RailwayReferenceCatalog:
    """Stable train/station/route reference loaded once from disk."""

    def __init__(self, path: str):
        self.path = Path(path)
        self.data: Dict[str, Any] = {}
        self.loaded = False
        self.error = None
        self.warnings = []
        self.system_knowledge = ""
        self.load()

    @staticmethod
    def _unwrap_export(raw):
        # Clean JSON
        if isinstance(raw, dict) and all(
            key in raw for key in ("trains", "stations", "routes")
        ):
            return raw

        # Neon result: [{"railway_catalog": "{...}"}]
        if (
            isinstance(raw, list)
            and len(raw) == 1
            and isinstance(raw[0], dict)
            and "railway_catalog" in raw[0]
        ):
            raw = raw[0]["railway_catalog"]

        # Wrapper: {"railway_catalog": ...}
        if isinstance(raw, dict) and "railway_catalog" in raw:
            raw = raw["railway_catalog"]

        if isinstance(raw, str):
            raw = json.loads(raw)

        if not isinstance(raw, dict):
            raise ValueError("Catalog root must resolve to a JSON object.")

        return raw

    def _repair_route_station_ids(self, raw: dict):
        """
        Repair a missing route-station station_id in memory by exact code/name.
        This does NOT write anything back to PostgreSQL.
        """
        stations = raw.get("stations", [])

        by_code = {
            _catalog_norm(s.get("code")): s.get("id")
            for s in stations
            if s.get("code") and s.get("id") is not None
        }
        by_name = {
            _catalog_norm(s.get("name")): s.get("id")
            for s in stations
            if s.get("name") and s.get("id") is not None
        }

        filled = 0

        for route in raw.get("routes", []):
            for rs in route.get("stations", []) or []:
                if rs.get("station_id") is not None:
                    continue

                sid = None
                if rs.get("station_code"):
                    sid = by_code.get(_catalog_norm(rs["station_code"]))
                if sid is None and rs.get("station_name"):
                    sid = by_name.get(_catalog_norm(rs["station_name"]))

                if sid is not None:
                    rs["station_id"] = sid
                    filled += 1

        if filled:
            self.warnings.append(
                f"Filled {filled} missing route station_id value(s) in memory."
            )

    def _build_system_knowledge(self, raw: dict) -> str:
        """
        Compact notation:
          S  = public stations.id
          RS = route_stations.id
          R  = routes.id
          T  = trains.id
          @  = station order on a route
          mi = miles from that route's origin

        Catalog-v2 example:
            RS450=S6@1|mi=0.0

        Backward-compatible catalog-v1 example:
            S6@1

        This is internal model context only and must never be shown
        to passengers.
        """
        station_parts = []

        for s in raw.get("stations", []):
            if s.get("id") is None:
                continue

            station_parts.append(
                f"S{s['id']}={s.get('name','')}|{s.get('code','')}|"
                f"city={s.get('city','')}|"
                f"region={s.get('state_region','')}"
            )

        route_lines = []

        for r in raw.get("routes", []):
            stops = []

            for rs in sorted(
                r.get("stations", []) or [],
                key=lambda x: x.get("order") or 10**9
            ):
                station_id = rs.get("station_id")
                route_station_id = rs.get("route_station_id")
                order = rs.get("order")
                distance_miles = rs.get("distance_from_origin")

                if station_id is None or order is None:
                    continue

                if route_station_id is not None:
                    stop = (
                        f"RS{route_station_id}"
                        f"=S{station_id}"
                        f"@{order}"
                    )
                else:
                    # Old catalog compatibility.
                    stop = f"S{station_id}@{order}"

                if distance_miles is not None:
                    stop += f"|mi={distance_miles}"

                stops.append(stop)

            route_lines.append(
                f"R{r.get('id')}={r.get('name','')}|"
                f"{r.get('origin','')}->{r.get('destination','')}|"
                f"mi={r.get('distance','')}|"
                f"dur={r.get('duration','')}|"
                + ",".join(stops)
            )

        train_parts = []

        for t in raw.get("trains", []):
            if t.get("id") is None:
                continue

            descriptor = _train_public_descriptor(
                t
            )

            train_entry = (
                f"T{t['id']}=No.{t.get('train_no','')}|"
                f"{t.get('train_name','')}|"
                f"R{t.get('route_id')}|"
                f"service={t.get('train_type','')}|"
                f"coaches={t.get('total_coaches','')}|"
                f"capacity={t.get('capacity','')}"
            )

            rolling_stock = descriptor.get(
                "rolling_stock_type"
            )

            if rolling_stock:
                train_entry += (
                    f"|stock={rolling_stock}"
                )

            aliases = descriptor.get(
                "aliases"
            ) or []

            if aliases:
                train_entry += (
                    "|aliases="
                    + ",".join(aliases)
                )

            service_tags = descriptor.get(
                "service_tags"
            ) or []

            if service_tags:
                train_entry += (
                    "|tags="
                    + ",".join(service_tags)
                )

            train_parts.append(
                train_entry
            )

        return (
            "STABLE RAILWAY REFERENCE — INTERNAL ONLY\n"
            "Notation: "
            "S=station_id, "
            "RS=route_station_id, "
            "R=route_id, "
            "T=train_id, "
            "@=route station order, "
            "mi=miles.\n"
            "Never reveal S/RS/R/T IDs, @ order, or internal notation "
            "to passengers.\n"
            "Use this only to understand entity identity/relationships "
            "and choose tool parameters.\n"
            "Train service type and rolling-stock type are different concepts: "
            "service=EXPRESS/LOCAL/etc. is backend train_type; "
            "stock=DEMU/etc. is passenger-facing rolling stock. "
            "A train can be service=EXPRESS and stock=DEMU at the same time.\n"
            "Passenger wording such as DEMU should match stock/aliases/tags, "
            "not be sent to the backend as train_type.\n"
            "For calculate_fare, RS values are RouteStation row IDs. "
            "Do NOT substitute public S values for RS values.\n"
            "Schedules, actual times, delays, live location, seats and "
            "bookings are NOT here.\n\n"
            "STATIONS:\n"
            + "; ".join(station_parts)
            + "\n\nROUTES:\n"
            + "\n".join(route_lines)
            + "\n\nTRAINS:\n"
            + "; ".join(train_parts)
        )


    def load(self):
        self.data = {}
        self.loaded = False
        self.error = None
        self.warnings = []
        self.system_knowledge = ""

        if not self.path.exists():
            self.error = f"Catalog file not found: {self.path}"
            print(f"⚠️ {self.error}")
            return self

        try:
            with self.path.open("r", encoding="utf-8") as f:
                raw = self._unwrap_export(json.load(f))

            for key in ("trains", "stations", "routes"):
                if key not in raw or not isinstance(raw[key], list):
                    raise ValueError(f"Catalog must contain a '{key}' list.")

            # Deep-copy into plain JSON-safe data.
            raw = json.loads(
                json.dumps(raw, ensure_ascii=False, default=str)
            )

            self._repair_route_station_ids(raw)

            # ----------------------------------------------------
            # Catalog-v2 diagnostics
            # ----------------------------------------------------
            route_station_rows = [
                rs
                for route in raw.get("routes", [])
                for rs in (route.get("stations", []) or [])
            ]

            rows_with_route_station_id = sum(
                1
                for rs in route_station_rows
                if rs.get("route_station_id") is not None
            )

            rows_with_distance = sum(
                1
                for rs in route_station_rows
                if rs.get("distance_from_origin") is not None
            )

            if route_station_rows:
                if rows_with_route_station_id == 0:
                    self.warnings.append(
                        "Catalog v1 detected: route_station_id is missing. "
                        "Fare questions will fall back to get_route before "
                        "calculate_fare. Regenerate catalog v2 to avoid that "
                        "extra lookup."
                    )
                elif rows_with_route_station_id < len(route_station_rows):
                    self.warnings.append(
                        "Some route stations are missing route_station_id. "
                        "Do not guess fare RouteStation IDs."
                    )

                if rows_with_distance == 0:
                    self.warnings.append(
                        "distance_from_origin is missing from the stable "
                        "catalog. Parcel calculation can still fetch route "
                        "mileage from the backend, but catalog v2 should "
                        "include this field in miles."
                    )

            self.data = raw
            self.system_knowledge = self._build_system_knowledge(raw)
            self.loaded = True

            print(
                "✅ Stable railway knowledge loaded once: "
                f"{len(raw['trains'])} trains, "
                f"{len(raw['stations'])} stations, "
                f"{len(raw['routes'])} routes"
            )
            print(
                "🧠 Persistent railway system knowledge size:",
                len(self.system_knowledge),
                "characters"
            )

            for warning in self.warnings:
                print(f"⚠️ Catalog: {warning}")

        except Exception as exc:
            self.error = f"Failed to load railway catalog: {exc}"
            print(f"⚠️ {self.error}")

        return self

    def reload(self):
        return self.load()


railway_catalog = RailwayReferenceCatalog(
    Config.RAILWAY_CATALOG_PATH
)

RAILWAY_SYSTEM_KNOWLEDGE = (
    railway_catalog.system_knowledge
    if railway_catalog.loaded
    else (
        "Stable railway reference is unavailable. "
        "Never guess an internal ID; use lookup tools when necessary."
    )
)

print("✅ Railway system knowledge ready")


✅ Stable railway knowledge loaded once: 6 trains, 25 stations, 4 routes
🧠 Persistent railway system knowledge size: 5725 characters
✅ Railway system knowledge ready


In [ ]:
# ============================================================
# v13 STABLE CATALOG v2 COMPATIBILITY SMOKE TEST
# ============================================================

_catalog_v2_test = {
    "stations": [
        {
            "id": 6,
            "code": "YGNMAIN023",
            "name": "ရန်ကုန် ဘူတာကြီး",
            "city": "ရန်ကုန်",
            "state_region": "ရန်ကုန်",
        },
        {
            "id": 7,
            "code": "BG-PYAY014",
            "name": "ပြည် ဘူတာ",
            "city": "ပြည်",
            "state_region": "ပဲခူး",
        },
    ],
    "routes": [
        {
            "id": 5,
            "name": "ရန်ကုန် ဘူတာကြီး - ပြည် ဘူတာ",
            "origin": "ရန်ကုန်မြို့",
            "destination": "ပြည်မြို့",
            "distance": 178.95,
            "duration": "8:30",
            "stations": [
                {
                    "route_station_id": 450,
                    "station_id": 6,
                    "station_code": "YGNMAIN023",
                    "station_name": "ရန်ကုန် ဘူတာကြီး",
                    "order": 1,
                    "distance_from_origin": 0.0,
                },
                {
                    "route_station_id": 474,
                    "station_id": 7,
                    "station_code": "BG-PYAY014",
                    "station_name": "ပြည် ဘူတာ",
                    "order": 25,
                    "distance_from_origin": 178.95,
                },
            ],
        }
    ],
    "trains": [
        {
            "id": 6,
            "route_id": 5,
            "train_no": "73",
            "train_name": "အမှတ်(၇၃)အဆန်",
            "train_type": "EXPRESS",
            "total_coaches": 6,
            "capacity": 290,
        }
    ],
}

_catalog_test_obj = RailwayReferenceCatalog.__new__(
    RailwayReferenceCatalog
)

_catalog_text = _catalog_test_obj._build_system_knowledge(
    _catalog_v2_test
)

assert "RS450=S6@1|mi=0.0" in _catalog_text
assert "RS474=S7@25|mi=178.95" in _catalog_text
assert "RS=route_station_id" in _catalog_text

print("✅ v13 catalog-v2 parser smoke test passed")


assert "service=EXPRESS" in _catalog_text
assert "stock=DEMU" in _catalog_text
assert "aliases=DEMU" in _catalog_text

print("✅ v16 DEMU passenger alias test passed")


✅ v13 catalog-v2 parser smoke test passed
✅ v16 DEMU passenger alias test passed


In [ ]:
# ============================================================
# v16 TRAIN TERMINOLOGY / DEMU RESOLUTION SMOKE TEST
# ============================================================

def stable_train_matches_passenger_term(
    term: str,
    catalog_data: dict
):
    """
    Resolve a passenger-visible train term against stable train metadata.

    This helper does not alter backend train_type.
    """
    needle = _catalog_norm(
        term
    )

    if not needle:
        return []

    matches = []

    for train in catalog_data.get(
        "trains",
        []
    ):
        descriptor = _train_public_descriptor(
            train
        )

        searchable = [
            train.get("train_no"),
            train.get("train_name"),
            train.get("train_type"),
            descriptor.get(
                "rolling_stock_type"
            ),
            *descriptor.get(
                "aliases",
                []
            ),
            *descriptor.get(
                "service_tags",
                []
            ),
        ]

        if any(
            needle in _catalog_norm(value)
            for value in searchable
            if value
        ):
            matches.append({
                "train_id": train.get("id"),
                "train_no": train.get("train_no"),
                "route_id": train.get("route_id"),
                "service_type": train.get("train_type"),
                "rolling_stock_type": (
                    descriptor.get(
                        "rolling_stock_type"
                    )
                ),
            })

    return matches


_v16_alias_catalog = {
    "trains": [
        {
            "id": 6,
            "train_no": "73",
            "train_name": "အမှတ်(၇၃)အဆန်",
            "route_id": 5,
            "train_type": "EXPRESS",
        },
        {
            "id": 9,
            "train_no": "74",
            "train_name": "အမှတ်(၇၄)အစုန်",
            "route_id": 14,
            "train_type": "EXPRESS",
        },
        {
            "id": 10,
            "train_no": "72",
            "train_name": "အမှတ်(၇၂)အစုန်",
            "route_id": 14,
            "train_type": "LOCAL",
        },
    ]
}

_demu_matches = stable_train_matches_passenger_term(
    "DEMU",
    _v16_alias_catalog
)

assert {
    item["train_no"]
    for item in _demu_matches
} == {"73", "74"}

assert all(
    item["service_type"] == "EXPRESS"
    for item in _demu_matches
)

assert all(
    item["rolling_stock_type"] == "DEMU"
    for item in _demu_matches
)

assert not any(
    item["train_no"] == "72"
    for item in _demu_matches
)

print("✅ v16 DEMU alias resolution smoke test passed")
print(_demu_matches)


✅ v16 DEMU alias resolution smoke test passed
[{'train_id': 6, 'train_no': '73', 'route_id': 5, 'service_type': 'EXPRESS', 'rolling_stock_type': 'DEMU'}, {'train_id': 9, 'train_no': '74', 'route_id': 14, 'service_type': 'EXPRESS', 'rolling_stock_type': 'DEMU'}]


## Optional future train fields in `railway_catalog.json`

You do **not** need to change PostgreSQL `train_type` from `EXPRESS` to `DEMU`.

A better stable catalog record is:

```json
{
  "id": 6,
  "train_no": "73",
  "train_name": "အမှတ်(၇၃)အဆန်",
  "route_id": 5,
  "train_type": "EXPRESS",
  "rolling_stock_type": "DEMU",
  "aliases": [
    "DEMU",
    "Diesel Electric Multiple Unit"
  ],
  "service_tags": [
    "Yangon-Pyay DEMU"
  ]
}
```

This preserves:

```text
train_type          = service classification
rolling_stock_type  = rolling-stock family
```

v16 already understands these optional fields. Until your catalog export includes
them, Train 73/74 use the notebook's stable DEMU fallback aliases.


## Refreshing the stable railway catalog after database changes

The chatbot notebook does **not** need a MongoDB knowledge-base rebuild when
stable PostgreSQL reference IDs change.

After changing routes, route stations, station IDs, train IDs, train-to-route
assignments, or `distance_from_origin`:

1. Run the catalog-v2 SQL export against PostgreSQL/Neon.
2. Replace the JSON file at `Config.RAILWAY_CATALOG_PATH`.
3. Run:

```python
refresh_railway_catalog()
```

Recommended route-station JSON shape:

```json
{
  "route_station_id": 450,
  "station_id": 6,
  "order": 1,
  "distance_from_origin": 0.0
}
```

`route_station_id` and `station_id` are intentionally different identifiers.

- `station_id` is used by station/schedule/parcel tools.
- `route_station_id` is used by the exact fare tool.
- `distance_from_origin` is in miles after your backend unit migration.


In [ ]:
from sentence_transformers import SentenceTransformer

class EmbeddingService:

    def __init__(self):

        print("Loading BGE-M3...")

        self.model = SentenceTransformer(
            Config.EMBEDDING_MODEL,
            device=Config.EMBEDDING_DEVICE
        )

        print("Embedding model ready.")

    def encode(self, text):

        return self.model.encode(
            text,
            normalize_embeddings=True
        ).tolist()

In [ ]:
from pymongo import MongoClient


class MongoService:

    def __init__(self):

        self.client = MongoClient(
            Config.MONGO_URI,
            serverSelectionTimeoutMS=10000
        )

        self.collection = self.client[
            Config.DATABASE_NAME
        ][
            Config.COLLECTION_NAME
        ]

        self.client.admin.command("ping")
        self.document_count = self.collection.count_documents({})

        print(
            "✅ MongoDB KB connected:",
            f"{Config.DATABASE_NAME}.{Config.COLLECTION_NAME}",
            f"({self.document_count} documents)"
        )

        if self.document_count == 0:
            print(
                "⚠️ Knowledge collection is empty. "
                "RAG knowledge questions will not be grounded."
            )

    def get_status(self):

        return {
            "database": Config.DATABASE_NAME,
            "collection": Config.COLLECTION_NAME,
            "documents": self.document_count,
            "vector_index": Config.VECTOR_INDEX,
        }

    def vector_search(self, embedding):

        pipeline = [

            {
                "$vectorSearch": {

                    "index": Config.VECTOR_INDEX,

                    "path": "embedding",

                    "queryVector": embedding,

                    # The Pyay-focused KB is larger than the original KB.
                    # A wider candidate pool gives BGE-M3 more room before
                    # the final TOP_K results are selected.
                    "numCandidates": 150,

                    "limit": Config.TOP_K

                }

            },

            {
                "$project": {

                    "_id": 0,

                    "dataset": 1,

                    "title": 1,

                    "content": 1,

                    "category": 1,

                    "language": 1,

                    "source_type": 1,

                    "source": 1,

                    "source_url": 1,

                    "as_of_date": 1,

                    "score": {
                        "$meta": "vectorSearchScore"
                    }

                }

            }

        ]

        return list(
            self.collection.aggregate(
                pipeline
            )
        )


In [ ]:
import logging

logger = logging.getLogger(__name__)

try:
    from deep_translator import GoogleTranslator
except ImportError:
    print("⚠️ deep-translator not installed. Run: !pip install -q deep-translator")
    GoogleTranslator = None


class DeepTranslationService:
    """
    Optional retrieval-fallback translator.

    IMPORTANT:
    This service is NOT used to translate every user message and it is
    NOT used to translate Qwen's final answer.

    It is loaded lazily only when BGE-M3's direct Myanmar retrieval
    confidence is weak.
    """

    def __init__(self):
        print("=" * 60)
        print("Loading Translation Fallback Service...")
        print("=" * 60)

        self.cache = {}
        self.total_translations = 0
        self.cache_hits = 0

        if GoogleTranslator is None:
            raise ImportError(
                "deep-translator is required for retrieval fallback. "
                "Install with: !pip install -q deep-translator"
            )

        self.mm_to_en_translator = GoogleTranslator(
            source="my",
            target="en"
        )

        print("✅ Translation fallback ready")
        print("   Used only for weak Myanmar RAG retrieval")
        print("=" * 60)

    def mm_to_en(self, text: str) -> str:
        """Translate Myanmar query to English for fallback retrieval only."""
        if not text or not text.strip():
            return ""

        cache_key = f"mm_to_en_{text}"

        if Config.ENABLE_CACHE and cache_key in self.cache:
            self.cache_hits += 1
            return self.cache[cache_key]

        try:
            self.total_translations += 1
            translated = self.mm_to_en_translator.translate(text)

            if Config.ENABLE_CACHE:
                self.cache[cache_key] = translated

            return translated or text

        except Exception as e:
            logger.error(f"Translation fallback error (my->en): {e}")
            return text

    def get_stats(self) -> dict:
        return {
            "total_translations": self.total_translations,
            "cache_hits": self.cache_hits,
            "cache_size": len(self.cache),
        }

    def clear_cache(self):
        self.cache.clear()

    def __del__(self):
        if hasattr(self, "cache"):
            self.cache.clear()


In [ ]:
# ============================================================
# PROVIDER-NEUTRAL RAILWAY TOOL DEFINITIONS — GEMINI + QWEN
# ============================================================
# Short schemas reduce Groq prompt/tool overhead. Stable IDs normally come
# from the relevant catalog slice; search tools are fallback lookups.

RAILWAY_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_stations",
            "description": "Fallback station lookup by passenger-visible name/code/city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "limit": {"type": "integer", "default": 10}
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_station",
            "description": "Get a station by internal stations.id.",
            "parameters": {
                "type": "object",
                "properties": {"station_id": {"type": "integer"}},
                "required": ["station_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_trains",
            "description": "List current train records.",
            "parameters": {
                "type": "object",
                "properties": {
                    "status": {"type": "string"},
                    "limit": {"type": "integer", "default": 50}
                },
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_trains",
            "description": "Fallback backend train lookup by train number/name. Resolve passenger rolling-stock wording such as DEMU from stable stock/aliases/tags first; DEMU is not necessarily a backend train_type. train_no is not train_id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "status": {"type": "string"},
                    "limit": {"type": "integer", "default": 10}
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_train",
            "description": "Get a train by internal trains.id.",
            "parameters": {
                "type": "object",
                "properties": {"train_id": {"type": "integer"}},
                "required": ["train_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_train_stops",
            "description": "Get ordered stops for internal train_id.",
            "parameters": {
                "type": "object",
                "properties": {"train_id": {"type": "integer"}},
                "required": ["train_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_route",
            "description": "Get route details by internal routes.id.",
            "parameters": {
                "type": "object",
                "properties": {"route_id": {"type": "integer"}},
                "required": ["route_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_trains_by_route",
            "description": "List trains assigned to a route.",
            "parameters": {
                "type": "object",
                "properties": {
                    "route_id": {"type": "integer"},
                    "status": {"type": "string"}
                },
                "required": ["route_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_schedules",
            "description": "AUTHORITATIVE passenger journey lookup. Returns matching dated services and station-specific expected departure/arrival times for the selected FROM/TO stations. Use this to resolve the exact schedule_id before checking seats. Do not use it as the final fare calculator when calculate_fare is available.",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_station_id": {"type": "integer"},
                    "to_station_id": {"type": "integer"},
                    "route_ids": {
                        "type": "array",
                        "items": {"type": "integer"},
                        "minItems": 1
                    },
                    "date_from": {"type": "string"},
                    "date_to": {"type": "string"}
                },
                "required": [
                    "from_station_id", "to_station_id",
                    "route_ids", "date_from", "date_to"
                ],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_route_train_schedule",
            "description": "Configured route/train stop schedule by internal route_id + train_id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "route_id": {"type": "integer"},
                    "train_id": {"type": "integer"}
                },
                "required": ["route_id", "train_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_next_train",
            "description": "Next arrival for route_id + train_id + route station order.",
            "parameters": {
                "type": "object",
                "properties": {
                    "route_id": {"type": "integer"},
                    "train_id": {"type": "integer"},
                    "station_order": {"type": "integer", "minimum": 1}
                },
                "required": ["route_id", "train_id", "station_order"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_fare",
            "description": "Calculate the exact backend-configured fare for one train and one ordered route-station pair. IMPORTANT: from_route_station_id and to_route_station_id are RouteStation row IDs. Use trusted RS values from the stable catalog v2 when available; otherwise resolve them with get_route. Never substitute public stations.id values. Defaults to ECONOMY_CLASS when class is not specified.",
            "parameters": {
                "type": "object",
                "properties": {
                    "train_id": {"type": "integer"},
                    "from_route_station_id": {"type": "integer"},
                    "to_route_station_id": {"type": "integer"},
                    "class_type": {
                        "type": "string",
                        "enum": [
                            "ECONOMY_CLASS",
                            "UPPER_CLASS",
                            "SLEEPER"
                        ],
                        "default": "ECONOMY_CLASS"
                    },
                    "seat_type": {"type": "string"}
                },
                "required": [
                    "train_id",
                    "from_route_station_id",
                    "to_route_station_id"
                ],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_fare_price_matrix",
            "description": "Get the backend-configured fare matrix for one train and current backend fare class. Valid classes are ECONOMY_CLASS, UPPER_CLASS and SLEEPER. Use for broader fare comparisons along a train route; never calculate fares yourself.",
            "parameters": {
                "type": "object",
                "properties": {
                    "train_id": {"type": "integer"},
                    "class_type": {
                        "type": "string",
                        "enum": [
                            "ECONOMY_CLASS",
                            "UPPER_CLASS",
                            "SLEEPER"
                        ],
                        "default": "ECONOMY_CLASS"
                    }
                },
                "required": ["train_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_parcel_fee",
            "description": "Calculate small-consignment parcel charge on supported Yangon-Pyay/Pyay-Yangon flat routes. Uses backend route-station distance_from_origin in miles. Args use public stations.id values. Rule: 13 pyas (0.13 MMK) per viss per mile, minimum 5 viss, minimum 100 miles, final fee rounded upward to the applicable 50-MMK boundary. Not for full-wagon freight.",
            "parameters": {
                "type": "object",
                "properties": {
                    "route_id": {"type": "integer"},
                    "from_station_id": {"type": "integer"},
                    "to_station_id": {"type": "integer"},
                    "weight_viss": {"type": "number", "exclusiveMinimum": 0}
                },
                "required": ["route_id", "from_station_id", "to_station_id", "weight_viss"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_schedule_seat_map",
            "description": "Get coaches and current seat availability for one exact dated service identified by schedule_id. Seat availability is schedule-specific.",
            "parameters": {
                "type": "object",
                "properties": {
                    "schedule_id": {"type": "integer"}
                },
                "required": ["schedule_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_seat_availability",
            "description": "Check whether one exact seat is currently available on one exact schedule.",
            "parameters": {
                "type": "object",
                "properties": {
                    "seat_id": {"type": "integer"},
                    "schedule_id": {"type": "integer"}
                },
                "required": ["seat_id", "schedule_id"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_ticket_journey_status",
            "description": "Read-only lookup for a passenger ticket number. Returns booking status and current schedule-scoped journey progress without exposing reservation/confirmation/cancellation write operations.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticket_no": {"type": "string"}
                },
                "required": ["ticket_no"],
                "additionalProperties": False
            }
        }
    },
]

print(f"✅ Railway tools loaded for Gemini/Qwen: {len(RAILWAY_TOOLS)}")


✅ Railway tools loaded for Gemini/Qwen: 17


In [ ]:
# ============================================================
# RAILWAY TOOL DISPATCH
# ============================================================
# Tool execution is handled safely inside:
#     RAGEngine.execute_railway_tool(...)
#
# The previous global dispatcher referenced undefined functions
# (search_stations / get_train_stops) during a fresh notebook run,
# so it is intentionally removed.


In [ ]:
# ============================================
# TOOL CALL PARSER
# ============================================

import re
import json


def parse_tool_call(text: str):
    """
    Extract a structured railway tool call from LLM output.

    Expected format:

    <TOOL_CALL>
    {"name":"search_stations","arguments":{"query":"Yangon"}}
    </TOOL_CALL>
    """

    pattern = r"<TOOL_CALL>\s*(.*?)\s*</TOOL_CALL>"

    match = re.search(
        pattern,
        text,
        re.DOTALL | re.IGNORECASE
    )

    if not match:
        return None

    raw_json = match.group(1).strip()

    try:
        tool_call = json.loads(raw_json)

        if not isinstance(tool_call, dict):
            return None

        if "name" not in tool_call:
            return None

        if "arguments" not in tool_call:
            tool_call["arguments"] = {}

        return tool_call

    except json.JSONDecodeError:
        return None

In [ ]:
# ============================================
# TOOL CALL VALIDATION (DEFENSE IN DEPTH)
# ============================================
# Groq validates the JSON schema, then we validate again before a
# backend request is executed.

from datetime import datetime

TOOL_SCHEMAS = {
    "search_stations": {"required": ["query"]},
    "get_station": {"required": ["station_id"]},
    "list_trains": {"required": []},
    "search_trains": {"required": ["query"]},
    "get_train": {"required": ["train_id"]},
    "get_train_stops": {"required": ["train_id"]},
    "get_route": {"required": ["route_id"]},
    "get_trains_by_route": {"required": ["route_id"]},
    "search_schedules": {
        "required": [
            "from_station_id", "to_station_id", "route_ids",
            "date_from", "date_to"
        ]
    },
    "get_route_train_schedule": {"required": ["route_id", "train_id"]},
    "get_next_train": {"required": ["route_id", "train_id", "station_order"]},
    "calculate_fare": {
        "required": [
            "train_id", "from_route_station_id", "to_route_station_id"
        ]
    },
    "get_fare_price_matrix": {"required": ["train_id"]},
    "calculate_parcel_fee": {
        "required": ["route_id", "from_station_id", "to_station_id", "weight_viss"]
    },
    "get_schedule_seat_map": {"required": ["schedule_id"]},
    "check_seat_availability": {"required": ["seat_id", "schedule_id"]},
    "get_ticket_journey_status": {"required": ["ticket_no"]},
}


def validate_tool_call(tool_call: dict):
    if not tool_call or not isinstance(tool_call, dict):
        return False, "Invalid tool call."

    tool_name = tool_call.get("name")
    arguments = tool_call.get("arguments", {})

    if tool_name not in TOOL_SCHEMAS:
        return False, f"Tool '{tool_name}' is not allowed."

    if not isinstance(arguments, dict):
        return False, "Tool arguments must be a JSON object."

    for argument in TOOL_SCHEMAS[tool_name]["required"]:
        if argument not in arguments:
            return False, f"Missing required argument '{argument}' for tool '{tool_name}'."

    if tool_name == "search_stations":
        query = arguments.get("query")
        if not isinstance(query, str) or not query.strip():
            return False, "search_stations requires a non-empty text query."
        arguments["query"] = query.strip()

    if tool_name == "search_trains":
        query = arguments.get("query")
        if isinstance(query, bool) or query is None:
            return False, "search_trains requires a train number or train name."
        if not isinstance(query, (str, int)):
            return False, "search_trains query must be text or an integer train reference."
        query = str(query).strip()
        if not query:
            return False, "search_trains requires a non-empty train reference."
        arguments["query"] = query

    if "limit" in arguments:
        try:
            limit = int(arguments["limit"])
        except (TypeError, ValueError):
            return False, "limit must be an integer."

        max_limit = 100 if tool_name == "list_trains" else 20
        if limit < 1 or limit > max_limit:
            return False, f"limit must be between 1 and {max_limit}."
        arguments["limit"] = limit

    if "status" in arguments and arguments["status"] is not None:
        status = str(arguments["status"]).strip()
        if not status:
            arguments.pop("status", None)
        else:
            arguments["status"] = status.upper()

    for name in {
        "station_id", "from_station_id", "to_station_id",
        "from_route_station_id", "to_route_station_id",
        "train_id", "route_id", "station_order",
        "schedule_id", "seat_id"
    }:
        if name in arguments:
            value = arguments[name]
            if isinstance(value, bool):
                return False, f"{name} must be an integer."
            try:
                value = int(value)
            except (TypeError, ValueError):
                return False, f"{name} must be an integer."
            if value <= 0:
                return False, f"{name} must be greater than zero."
            arguments[name] = value

    if "ticket_no" in arguments:
        ticket_no = arguments.get("ticket_no")
        if not isinstance(ticket_no, str) or not ticket_no.strip():
            return False, "ticket_no must be a non-empty string."
        arguments["ticket_no"] = ticket_no.strip()

    if "class_type" in arguments and arguments["class_type"] is not None:

        allowed_classes = {
            "ECONOMY_CLASS",
            "UPPER_CLASS",
            "SLEEPER",
        }

        aliases = {
            "ECONOMY": "ECONOMY_CLASS",
            "ECONOMY CLASS": "ECONOMY_CLASS",
            "ECONOMY_CLASS": "ECONOMY_CLASS",

            # Compatibility with older wording.
            "ORDINARY": "ECONOMY_CLASS",
            "ORDINARY CLASS": "ECONOMY_CLASS",
            "ORDINARY_CLASS": "ECONOMY_CLASS",

            "UPPER": "UPPER_CLASS",
            "UPPER CLASS": "UPPER_CLASS",
            "UPPER_CLASS": "UPPER_CLASS",

            "SLEEPER": "SLEEPER",
            "SLEEPER CLASS": "SLEEPER",
            "SLEEPER_CLASS": "SLEEPER",
        }

        raw_class = str(arguments["class_type"]).strip().upper()
        class_type = aliases.get(raw_class, raw_class)

        if class_type not in allowed_classes:
            return False, (
                "class_type must be one of: "
                "ECONOMY_CLASS, UPPER_CLASS, SLEEPER"
            )

        arguments["class_type"] = class_type

    if "seat_type" in arguments and arguments["seat_type"] is not None:
        seat_type = str(arguments["seat_type"]).strip()
        arguments["seat_type"] = seat_type or None

    if "route_ids" in arguments:
        route_ids = arguments["route_ids"]
        if not isinstance(route_ids, list) or not route_ids:
            return False, "route_ids must be a non-empty array of route IDs."

        normalized = []
        for value in route_ids:
            if isinstance(value, bool):
                return False, "Every route_ids value must be an integer."
            try:
                route_id = int(value)
            except (TypeError, ValueError):
                return False, "Every route_ids value must be an integer."
            if route_id <= 0:
                return False, "Every route_ids value must be greater than zero."
            if route_id not in normalized:
                normalized.append(route_id)

        arguments["route_ids"] = normalized

    for field in ("date_from", "date_to"):
        if field in arguments:
            value = str(arguments[field]).strip()
            try:
                datetime.strptime(value, "%Y-%m-%d")
            except ValueError:
                return False, f"{field} must use YYYY-MM-DD format."
            arguments[field] = value

    if "date_from" in arguments and "date_to" in arguments:
        if arguments["date_from"] > arguments["date_to"]:
            return False, "date_from must be on or before date_to."

    if tool_name == "calculate_parcel_fee":
        for key in ("route_id", "from_station_id", "to_station_id"):
            try:
                arguments[key] = int(arguments[key])
            except (TypeError, ValueError):
                return False, f"{key} must be an integer."
            if arguments[key] <= 0:
                return False, f"{key} must be greater than zero."

        try:
            arguments["weight_viss"] = float(arguments["weight_viss"])
        except (TypeError, ValueError):
            return False, "weight_viss must be a number."

        if arguments["weight_viss"] <= 0:
            return False, "weight_viss must be greater than zero."

    return True, None


In [ ]:
# ============================================
# LLM-SAFE TOOL PAYLOAD COMPACTION
# ============================================

def compact_tool_result_for_llm(
    tool_name: str,
    result: dict
) -> dict:

    if not isinstance(result, dict):
        return {
            "success": False,
            "error": "Backend returned an invalid result."
        }

    if not result.get("success"):
        return result

    if tool_name != "get_schedule_seat_map":
        return result

    data = result.get("data")

    if isinstance(data, dict):
        coaches = data.get("coaches") or []
        top_level = {
            key: value
            for key, value in data.items()
            if key not in {"coaches", "seats"}
        }
    elif isinstance(data, list):
        coaches = data
        top_level = {}
    else:
        return result

    summaries = []
    total_seats = 0
    available_seats = 0

    for coach in coaches:

        if not isinstance(coach, dict):
            continue

        seats = coach.get("seats") or []

        if not isinstance(seats, list):
            seats = []

        available = [
            seat
            for seat in seats
            if (
                isinstance(seat, dict)
                and seat.get("available") is True
            )
        ]

        total_seats += len(seats)
        available_seats += len(available)

        summaries.append({
            "coach_id": coach.get("id"),
            "coach_name": coach.get("name"),
            "coach_type": coach.get("coach_type"),
            "order_number": coach.get("order_number"),
            "total_seats": len(seats),
            "available_seats": len(available),
            "available_seat_examples": [
                seat.get("seat_number")
                for seat in available[:8]
                if seat.get("seat_number") is not None
            ],
        })

    return {
        "success": True,
        "data": {
            **top_level,
            "total_coaches": len(summaries),
            "total_seats": total_seats,
            "available_seats": available_seats,
            "coaches": summaries,
            "note": (
                "Full seat rows compacted for LLM context. "
                "Use check_seat_availability for one exact seat."
            ),
        }
    }


# ============================================
# TOOL RESULT FORMATTER
# ============================================

def format_tool_result(
    tool_name: str,
    result: dict
) -> str:

    train_tools = {
        "list_trains",
        "search_trains",
        "get_train",
        "get_trains_by_route"
    }

    if tool_name == "calculate_parcel_fee":
        return f"""
AUTHORITATIVE YANGON-PYAY PARCEL CALCULATION

This result uses backend route-station mileage plus the official Myanma Railways
flat-route small-consignment rule.

Rules:
- 13 pyas = 0.13 MMK per viss per mile
- minimum chargeable weight = 5 viss
- minimum chargeable distance = 100 miles
- final amount is rounded upward to the applicable 50-MMK boundary
- do not expose internal route/station IDs
- not for full-wagon freight or mountain-route pricing

CALCULATION DATA:
{json.dumps(result, ensure_ascii=False, indent=2, default=str)}
"""

    if tool_name in train_tools:

        return f"""
LIVE RAILWAY DATABASE RESULT

Tool: {tool_name}

The data below came directly from the Railway FastAPI backend.

IMPORTANT:
- Treat these values as authoritative.
- Do NOT invent missing values.
- Do NOT change train_no.
- Do NOT change train_name.
- Do NOT change train_type.
- Do NOT change route_id.
- Do NOT change total_coaches.
- Do NOT change capacity.
- Do NOT change speed.
- Do NOT change status.
- Do NOT confuse train_id with train_no.
- Answer the passenger using only the relevant fields.
- Do not expose raw JSON unless the passenger specifically asks for it.

DATABASE DATA:

{json.dumps(
    result,
    ensure_ascii=False,
    indent=2,
    default=str
)}
"""

    return f"""
Tool: {tool_name}

Result:
{json.dumps(
    result,
    ensure_ascii=False,
    indent=2,
    default=str
)}
"""


In [ ]:
# ============================================
# MULTILINGUAL PROMPT BUILDER — v7
# ============================================
# Stable railway IDs/relationships are now provider SYSTEM knowledge.
# This prompt contains only changing per-turn context:
# current date, recent conversation, RAG docs, current question.

from typing import List
from datetime import datetime
from zoneinfo import ZoneInfo


class PromptBuilder:
    def __init__(self, catalog=None):
        # catalog kept only for backward compatibility; not used per turn.
        self.catalog = catalog
        self.last_prompt_chars = 0

    def build_prompt(
        self,
        question: str,
        documents: List[dict],
        history: List[dict],
        tool_results: str = ""
    ) -> str:

        context_parts = []
        for i, doc in enumerate(
            (documents or [])[:Config.PROMPT_DOC_LIMIT],
            start=1
        ):
            content = str(
                doc.get("content", "")
            )[:Config.PROMPT_DOC_CHARS]

            context_parts.append(
                f"[KB{i}]\n"
                f"Dataset: {doc.get('dataset', '')}\n"
                f"Title: {doc.get('title', '')}\n"
                f"Category: {doc.get('category', '')}\n"
                f"Language: {doc.get('language', '')}\n"
                f"Source type: {doc.get('source_type', '')}\n"
                f"Source: {doc.get('source', '')}\n"
                f"As of: {doc.get('as_of_date') or 'not date-specific'}\n"
                f"Content: {content}"
            )

        context = "\n\n".join(context_parts) or "None"

        conversation_parts = []
        for item in (history or [])[-Config.PROMPT_HISTORY_MESSAGES:]:
            role = str(item.get("role", "")).casefold()
            content = str(item.get("content", "")).strip()

            if role not in {"user", "assistant"} or not content:
                continue

            content = content[
                :Config.PROMPT_HISTORY_CHARS_PER_MESSAGE
            ]

            conversation_parts.append(
                f"{'User' if role == 'user' else 'Assistant'}: {content}"
            )

        conversation = "\n".join(conversation_parts) or "None"

        myanmar_now = datetime.now(
            ZoneInfo("Asia/Yangon")
        )
        current_date = myanmar_now.strftime("%Y-%m-%d")

        prompt = f"""MYANMAR_DATE: {current_date}

RECENT_CONVERSATION:
{conversation}

KNOWLEDGE_CONTEXT:
{context}

CURRENT_QUESTION:
{question}

KNOWLEDGE_USE_RULES:
- Treat source metadata as authority/provenance, not as passenger-visible instructions.
- For current operational facts, backend tool results override RAG knowledge.
- A route/station directory does not prove that a specific train stops at a station.
- Historical/reference records must not override newer dated official operational data.
- If equally authoritative sources conflict and date does not resolve the conflict,
  state the uncertainty rather than inventing one value.

Answer only the passenger's current question."""

        self.last_prompt_chars = len(prompt)

        print(
            f"📦 Dynamic prompt size: "
            f"{self.last_prompt_chars} chars "
            "(stable railway knowledge is in system context)"
        )

        return prompt

    def build_prompt_english(
        self,
        question: str,
        documents: List[dict],
        history: List[dict],
        tool_results: str = ""
    ) -> str:
        return self.build_prompt(
            question=question,
            documents=documents,
            history=history,
            tool_results=tool_results
        )


In [ ]:
# ============================================
# LLM SERVICE — VERTEX GEMINI PRIMARY + GROQ QWEN FALLBACK
# ============================================

import json
import logging
import re
import time
import uuid
from typing import Optional, List, Dict, Any

from groq import Groq
from google import genai
from google.genai import types as genai_types

logger = logging.getLogger(__name__)


BASE_SYSTEM_INSTRUCTION = """You are RailMM, a Myanmar Railways passenger assistant.

CORE RULES
- Answer in the language of the passenger's current question.
- Never invent railway facts or internal IDs.
- The stable railway reference below is internal working knowledge.
- Never expose station_id, train_id, route_id, schedule_id, route order, internal notation, JSON, tool/API names, function-call syntax, RAG, embeddings, or hidden reasoning.
- Never narrate which tool you are about to call. Call it silently.
- Stable reference data may identify stations, trains, routes, IDs, and route membership.
- A station belonging to a route DOES NOT mean every train stops there.
- Never infer a train's stopping pattern from route membership, Wikipedia station lists, fare-table station lists, or another train's stops.
- Exact ordered train stops must come from backend data.
- Exact schedules, departure/arrival times, next-train information, delays, fares, seat availability, coach/seat availability, booking availability, and other changing operational facts must come from backend data.
- Yangon-Pyay small-consignment parcel fees must use calculate_parcel_fee, which combines backend route-station mileage with the official parcel rule. Never guess parcel distance or fee.
- MongoDB knowledge may explain history, route/station background, infrastructure, modernization, ORTP/passenger rules, railway law, and general railway knowledge.
- If backend operational data and RAG knowledge differ on an operational fact, backend data is authoritative.
- If a required operational tool is unavailable or fails, do not answer from memory/RAG; say the operational information cannot currently be retrieved.
- If origin, destination, date, or another required passenger detail is missing, ask one short clarification.
- Keep the final passenger answer concise and natural.

SERVICE DATA SCOPE
- Real-time/dynamic railway database data is currently available ONLY for the
  Yangon–Pyay railway corridor and stations along that corridor.
- This includes supported Yangon -> Pyay and Pyay -> Yangon services and the
  corresponding route/station records present in the stable railway reference.
- Only use railway schedule/live backend tools when the requested journey is
  covered by the supported Yangon–Pyay corridor.
- Do NOT claim that live schedules, current train times, delays, seat
  availability, or other dynamic information are available for routes outside
  the supported Yangon–Pyay corridor.
- For routes outside the supported Yangon–Pyay corridor:
  1. Answer using the retrieved RAG documents when they contain relevant
     information.
  2. Clearly state that live database information is not available for that
     route in this project.
  3. For the latest official schedule/service information, advise the passenger
     to check the official Myanmar Railways website:
     https://www.railways.gov.mm/
- Do not invent schedules or operational data for unsupported routes.
- If the RAG documents do not contain enough information about an unsupported
  route, say that the available project information is insufficient and refer
  the passenger to the official Myanmar Railways website.

YANGON-PYAY DEMO RULES
- Trains 63/64, 71/72 and 73/74 can have different stopping patterns.
- Use backend data for the stops of each train.
- Use backend data for Yangon-Pyay schedules, exact fares, and seat availability.
- Do not use the physical Yangon-Pyay station list as a train-specific stop list.

SOURCE PRECEDENCE
- For current operational facts, prefer the newest dated official backend/source information.
- Undated official/reference knowledge must not override newer dated official operational information.
- Historical records are historical context, not current operational truth.
- When equally authoritative sources conflict and date/authority does not resolve the conflict, acknowledge the conflict instead of inventing a single value.

KNOWLEDGE QUESTION ROUTING
- Passenger rules, NRC/passport/foreigner guidance, child-ticket rules,
  baggage, pets, ORTP, e-Receipt rules, refunds, service fees, complaints,
  official contacts, passenger insurance, railway law, history,
  infrastructure and station background are MongoDB RAG knowledge questions.
- Do not call station/train/schedule/fare/seat tools merely because RAG
  retrieval is weak.
- If no verified RAG knowledge is available for a knowledge question,
  say the project knowledge base does not contain enough verified information
  instead of inventing an answer or misusing an operational API.

TRAIN TERMINOLOGY / ALIAS ROUTING
- Keep backend train_type separate from passenger rolling-stock terminology.
- EXPRESS, LOCAL, etc. are service classifications stored by the backend.
- DEMU means Diesel Electric Multiple Unit and is rolling-stock terminology, not a backend train_type enum.
- A train may therefore be both EXPRESS and DEMU. These labels are not contradictory.
- In the current Yangon–Pyay stable reference, Train 73 and Train 74 may be identified by passengers as DEMU trains.
- If the passenger says "DEMU train", "DEMU Express", "diesel-electric multiple unit", or equivalent wording, resolve the train from stock/aliases/tags in stable railway knowledge.
- Direction still matters: Yangon -> Pyay and Pyay -> Yangon may refer to different train numbers.
- Do not automatically label another train as DEMU unless trusted stable catalog metadata or verified retrieved knowledge says so.
- Do not send DEMU as a backend train_type filter unless a future backend explicitly defines such a field.

TOOL ROUTING
- train_no is NOT train_id.
- Use the stable reference to choose known station/train/route IDs.
- If an entity is genuinely not in stable reference, use search_stations/search_trains.
- For A -> B schedules, choose only routes where origin order < destination order, then call search_schedules.
- search_schedules dates use YYYY-MM-DD.
- get_next_train requires route_id, train_id and station_order.
- For train-stop questions, use the backend train-stop tool; never infer stops from route membership.
- For Yangon-Pyay/Pyay-Yangon small-consignment parcel-fee questions, use calculate_parcel_fee.
- Parcel calculation requires origin, destination and weight in viss; if weight is missing, ask one short clarification.
- Supported flat-route rule: 13 pyas = 0.13 MMK per viss per mile; minimum chargeable weight 5 viss; minimum chargeable distance 100 miles; round final fee upward to the applicable 50-MMK boundary.
- Do not use this tool for full-wagon freight or routes outside Yangon-Pyay/Pyay-Yangon.
- For exact fare questions, use calculate_fare. Never compute the fare yourself from base fare, distance, rate, or surcharge fields.
- calculate_fare requires RouteStation row IDs. Its from_route_station_id/to_route_station_id are NOT public station_id values.
- In stable catalog v2, notation RS<number> is the trusted RouteStation row ID for that station on that route. Use the correct RS value directly when it is present.
- If the stable catalog does not contain the required RS values, call get_route before calculate_fare. Never guess an RS value and never substitute a public S/station_id value.
- The CURRENT backend fare classes are ECONOMY_CLASS, UPPER_CLASS, and SLEEPER. Passenger wording Economy or legacy Ordinary maps to ECONOMY_CLASS. Upper Class maps to UPPER_CLASS. If no class is specified, use ECONOMY_CLASS. Never send ORDINARY, FIRST_CLASS, AC_CHAIR or AC_SLEEPER to the current fare API.
- For seat/coach availability, first resolve the exact dated service with search_schedules, then call get_schedule_seat_map using that schedule_id.
- A seat being available on one schedule says nothing about another schedule. Never reuse seat availability across dates/services.
- Use check_seat_availability only after a concrete seat and schedule are known.
- For ticket or booking journey-status questions, use get_ticket_journey_status with the passenger-provided ticket number.
- Booking tools are read-only in RailMM. Never attempt to reserve, confirm, cancel, refund, or change a booking. Direct the passenger to the normal booking interface for such actions.
- Do not request or expose NRC, phone, email, payment details, or other passenger personal data in chatbot tool calls.
- Never guess an ID.
- For ANY question asking the departure or arrival time between specific stations, search_schedules is the authoritative source.
- Use only explicit station-specific departure_time/arrival_time (or explicit expected/actual timing fields) returned for the selected station pair.
- Never derive an intermediate-station time from route duration, route order, time_from_origin_minutes, or the route's overall schedule start/end time.
- If the exact station-specific time is absent, say it is not available; do not estimate.
- For booking availability, combine only current backend facts: the exact schedule must be bookable/current, seats must be available for that schedule, and fare must be backend-configured. Do not infer availability from train capacity alone.
- Before calling journey-search, fare, or seat tools, verify that the requested
  origin and destination belong to the supported Yangon–Pyay corridor.
- get_ticket_journey_status is the exception: when the passenger supplies a ticket
  number, call it directly. The booking itself identifies its schedule and segment.
- If the journey is outside the supported Yangon–Pyay corridor, do NOT call
  dynamic railway tools. Use RAG knowledge instead and refer the passenger to
  https://www.railways.gov.mm/ for current official information.
"""


class HybridLLMService:
    """
    Provider-neutral LLM service.

    Model modes:
      Auto   -> Gemini first, Qwen only if Gemini request fails.
      Gemini -> Gemini only.
      Qwen   -> Qwen only.

    All provider responses are normalized to:
      {
        "text": str,
        "tool_calls": [{"id", "name", "arguments"}],
        "provider": "gemini" | "qwen",
        "model": str,
      }
    """

    MODEL_MODE_ALIASES = {
        "auto": "Auto",
        "auto (gemini → qwen)": "Auto",
        "auto (gemini -> qwen)": "Auto",
        "gemini": "Gemini",
        "qwen": "Qwen",
    }

    def __init__(self, stable_knowledge: str = ""):
        print("=" * 60)
        print("Initializing Gemini-primary hybrid LLM service...")
        print("=" * 60)

        self.stable_knowledge = stable_knowledge or ""
        self.system_instruction = self._compose_system_instruction()

        self.total_llm_calls = 0
        self.total_llm_time = 0
        self.total_api_attempts = 0

        self.last_model_used = None
        self.last_provider_used = None
        self.last_usage = {}
        self.last_error = None

        # ---------------- Qwen / Groq ----------------
        self.qwen_client = Groq(
            api_key=Config.GROQ_API_KEY,
            timeout=Config.LLM_TIMEOUT,
        )
        self.qwen_available = True

        # ---------------- Gemini / Vertex AI ----------------
        self.gemini_client = None
        self.gemini_available = False
        self.gemini_error = None
        self.vertex_project_id = (
            Config.GOOGLE_CLOUD_PROJECT_ID or ""
        ).strip()

        self._initialize_gemini()

        # Convert the same provider-neutral tool schemas once.
        self._gemini_tool = self._build_gemini_tool(
            RAILWAY_TOOLS
        )

        print(
            "✅ Gemini:",
            (
                f"{Config.GEMINI_MODEL} via Vertex AI"
                if self.gemini_available
                else f"unavailable ({self.gemini_error})"
            )
        )
        print("✅ Qwen fallback:", Config.QWEN_MODEL, "via Groq")
        print("✅ Default:", Config.DEFAULT_MODEL_MODE)
        print("=" * 60)

    def _compose_system_instruction(self) -> str:
        return (
            BASE_SYSTEM_INSTRUCTION
            + "\n\n"
            + (self.stable_knowledge or "")
        )

    def update_stable_knowledge(self, stable_knowledge: str):
        self.stable_knowledge = stable_knowledge or ""
        self.system_instruction = self._compose_system_instruction()
        print(
            "🧠 LLM stable railway system knowledge updated:",
            len(self.stable_knowledge),
            "chars"
        )

    def _initialize_gemini(self):
        try:
            if not self.vertex_project_id:
                # Same Colab workflow used by the inspection notebook:
                # first try Colab Secret, then ask once.
                try:
                    from google.colab import userdata
                    value = userdata.get(
                        "GOOGLE_CLOUD_PROJECT_ID"
                    )
                    if value:
                        self.vertex_project_id = str(
                            value
                        ).strip()
                except Exception:
                    pass

            if not self.vertex_project_id:
                self.vertex_project_id = input(
                    "Enter Google Cloud PROJECT ID for Vertex AI "
                    "(leave blank to use Qwen only): "
                ).strip()

            if not self.vertex_project_id:
                self.gemini_error = (
                    "Google Cloud project ID was not provided."
                )
                return

            # Colab interactive Google account authentication.
            try:
                from google.colab import auth
                auth.authenticate_user(
                    project_id=self.vertex_project_id
                )
            except ImportError:
                # Outside Colab, google-genai can use ADC.
                pass

            self.gemini_client = genai.Client(
                vertexai=True,
                project=self.vertex_project_id,
                location=Config.GEMINI_LOCATION,
                http_options=genai_types.HttpOptions(
                    api_version="v1"
                ),
            )

            probe = self.gemini_client.models.generate_content(
                model=Config.GEMINI_MODEL,
                contents="Reply with exactly OK.",
                config=genai_types.GenerateContentConfig(
                    temperature=0,
                    max_output_tokens=32,
                    thinking_config=genai_types.ThinkingConfig(
                        thinking_budget=0
                    ),
                ),
            )

            self.gemini_available = bool(
                (probe.text or "").strip()
            )

        except Exception as exc:
            self.gemini_available = False
            self.gemini_error = (
                f"{type(exc).__name__}: {exc}"
            )
            logger.warning(
                "Vertex Gemini unavailable; Auto mode will use Qwen: %s",
                exc,
            )

    @staticmethod
    def _strip_schema_for_gemini(schema: dict) -> dict:
        """Keep standard JSON Schema fields accepted by parameters_json_schema."""
        if not isinstance(schema, dict):
            return schema

        result = {}
        for key, value in schema.items():
            if key in {"default", "additionalProperties"}:
                continue
            if isinstance(value, dict):
                result[key] = (
                    HybridLLMService._strip_schema_for_gemini(
                        value
                    )
                )
            elif isinstance(value, list):
                result[key] = [
                    (
                        HybridLLMService._strip_schema_for_gemini(v)
                        if isinstance(v, dict)
                        else v
                    )
                    for v in value
                ]
            else:
                result[key] = value
        return result

    def _build_gemini_tool(self, tools: list):
        declarations = []

        for tool in tools:
            f = tool["function"]

            declarations.append(
                genai_types.FunctionDeclaration(
                    name=f["name"],
                    description=f.get("description", ""),
                    parameters_json_schema=(
                        self._strip_schema_for_gemini(
                            f.get(
                                "parameters",
                                {"type": "object"}
                            )
                        )
                    ),
                )
            )

        return genai_types.Tool(
            function_declarations=declarations
        )

    @classmethod
    def normalize_model_mode(cls, mode: str) -> str:
        if not mode:
            return "Auto"
        return cls.MODEL_MODE_ALIASES.get(
            str(mode).strip().casefold(),
            "Auto"
        )

    def choose_provider(self, model_mode: str) -> str:
        mode = self.normalize_model_mode(model_mode)

        if mode == "Gemini":
            if not self.gemini_available:
                raise RuntimeError(
                    "Gemini was selected but Vertex AI Gemini "
                    f"is unavailable: {self.gemini_error}"
                )
            return "gemini"

        if mode == "Qwen":
            return "qwen"

        # Auto
        if self.gemini_available:
            return "gemini"
        return "qwen"

    @staticmethod
    def fallback_provider(
        provider: str,
        model_mode: str
    ) -> Optional[str]:
        mode = HybridLLMService.normalize_model_mode(
            model_mode
        )
        if mode == "Auto" and provider == "gemini":
            return "qwen"
        return None

    @staticmethod
    def _clean_response(text: str) -> str:
        if not text:
            return ""

        text = re.sub(
            r"<think>.*?</think>",
            "",
            text,
            flags=re.DOTALL | re.IGNORECASE,
        )

        for token in [
            "<|im_start|>",
            "<|im_end|>",
            "<|endoftext|>",
            "</s>",
        ]:
            text = text.replace(token, "")

        return text.strip()

    # ============================================================
    # NORMALIZED MESSAGE -> GROQ
    # ============================================================

    def _messages_for_qwen(self, messages: list):
        qwen_messages = [
            {
                "role": "system",
                "content": self.system_instruction,
            }
        ]

        for message in messages:
            role = message["role"]

            if role in {"user", "system"}:
                qwen_messages.append({
                    "role": role,
                    "content": message.get("content", ""),
                })

            elif role == "assistant":
                item = {
                    "role": "assistant",
                    "content": message.get("content") or None,
                }

                calls = message.get("tool_calls") or []
                if calls:
                    item["tool_calls"] = [
                        {
                            "id": call["id"],
                            "type": "function",
                            "function": {
                                "name": call["name"],
                                "arguments": json.dumps(
                                    call.get(
                                        "arguments",
                                        {}
                                    ),
                                    ensure_ascii=False,
                                ),
                            },
                        }
                        for call in calls
                    ]

                qwen_messages.append(item)

            elif role == "tool":
                qwen_messages.append({
                    "role": "tool",
                    "tool_call_id": message["tool_call_id"],
                    "name": message["name"],
                    "content": message.get("content", ""),
                })

        return qwen_messages

    # ============================================================
    # NORMALIZED MESSAGE -> GEMINI
    # ============================================================

    def _messages_for_gemini(self, messages: list):
        """
        Convert normalized history to Gemini Content objects.

        Gemini 3/3.5 function calls can carry an opaque thought_signature.
        If a model turn originated from Gemini, keep the exact SDK Content
        object returned by Gemini instead of rebuilding only name/args.
        """
        contents = []

        for message in messages:
            role = message["role"]

            if role == "system":
                continue

            if role == "user":
                contents.append(
                    genai_types.Content(
                        role="user",
                        parts=[
                            genai_types.Part.from_text(
                                text=message.get(
                                    "content",
                                    ""
                                )
                            )
                        ],
                    )
                )

            elif role == "assistant":
                gemini_content = message.get(
                    "gemini_content"
                )

                # Exact response part from Gemini: preserves thought_signature.
                if gemini_content is not None:
                    contents.append(gemini_content)
                    continue

                # Reconstruction is only for assistant history that did not
                # originate from the current Gemini function-calling step.
                parts = []

                content = message.get("content")
                if content:
                    parts.append(
                        genai_types.Part.from_text(
                            text=content
                        )
                    )

                for call in (
                    message.get("tool_calls") or []
                ):
                    parts.append(
                        genai_types.Part.from_function_call(
                            name=call["name"],
                            args=call.get(
                                "arguments",
                                {}
                            ),
                        )
                    )

                if parts:
                    contents.append(
                        genai_types.Content(
                            role="model",
                            parts=parts,
                        )
                    )

            elif role == "tool":
                try:
                    response_obj = json.loads(
                        message.get("content", "{}")
                    )
                except Exception:
                    response_obj = {
                        "result": message.get(
                            "content",
                            ""
                        )
                    }

                function_response_part = (
                    genai_types.Part.from_function_response(
                        name=message["name"],
                        response=response_obj,
                    )
                )

                # Gemini function calling expects the function response after
                # the model's preserved functionCall turn.
                contents.append(
                    genai_types.Content(
                        role="user",
                        parts=[function_response_part],
                    )
                )

        return contents

    # ============================================================
    # PROVIDER REQUESTS
    # ============================================================

    def _qwen_turn(
        self,
        messages: list,
        tools: list,
        tool_choice: str,
        max_new_tokens: int
    ) -> dict:
        qwen_messages = self._messages_for_qwen(
            messages
        )

        completion = self.qwen_client.chat.completions.create(
            model=Config.QWEN_MODEL,
            messages=qwen_messages,
            tools=tools,
            tool_choice=tool_choice,
            parallel_tool_calls=False,
            temperature=Config.LLM_TEMPERATURE,
            top_p=Config.LLM_TOP_P,
            max_completion_tokens=max_new_tokens,
            reasoning_effort=Config.QWEN_REASONING_EFFORT,
        )

        message = completion.choices[0].message

        tool_calls = []
        for call in list(message.tool_calls or []):
            try:
                arguments = json.loads(
                    call.function.arguments or "{}"
                )
            except Exception:
                arguments = {}

            tool_calls.append({
                "id": call.id or str(uuid.uuid4()),
                "name": call.function.name,
                "arguments": arguments,
            })

        usage = getattr(completion, "usage", None)
        if usage is None:
            usage_dict = {}
        elif hasattr(usage, "model_dump"):
            usage_dict = usage.model_dump()
        else:
            usage_dict = {"raw": str(usage)}

        return {
            "text": self._clean_response(
                message.content or ""
            ),
            "tool_calls": tool_calls,
            "provider": "qwen",
            "model": (
                getattr(completion, "model", None)
                or Config.QWEN_MODEL
            ),
            "usage": usage_dict,
        }

    def _gemini_turn(
        self,
        messages: list,
        tools: list,
        tool_choice: str,
        max_new_tokens: int
    ) -> dict:
        if not self.gemini_available:
            raise RuntimeError(
                f"Gemini is unavailable: {self.gemini_error}"
            )

        mode_map = {
            "auto": "AUTO",
            "required": "ANY",
            "none": "NONE",
        }
        fc_mode = mode_map.get(
            tool_choice,
            "AUTO"
        )

        config_kwargs = {
            "system_instruction": self.system_instruction,
            "temperature": Config.LLM_TEMPERATURE,
            "top_p": Config.LLM_TOP_P,
            "max_output_tokens": max_new_tokens,
            "thinking_config": genai_types.ThinkingConfig(
                thinking_budget=0
            ),
        }

        # Function calling is manual so our FastAPI validation/dispatcher
        # stays identical for Gemini and Qwen.
        if fc_mode != "NONE":
            config_kwargs.update({
                "tools": [self._gemini_tool],
                "automatic_function_calling": (
                    genai_types.AutomaticFunctionCallingConfig(
                        disable=True
                    )
                ),
                "tool_config": genai_types.ToolConfig(
                    function_calling_config=(
                        genai_types.FunctionCallingConfig(
                            mode=fc_mode
                        )
                    )
                ),
            })

        response = self.gemini_client.models.generate_content(
            model=Config.GEMINI_MODEL,
            contents=self._messages_for_gemini(
                messages
            ),
            config=genai_types.GenerateContentConfig(
                **config_kwargs
            ),
        )

        tool_calls = []
        for call in list(
            getattr(response, "function_calls", None)
            or []
        ):
            tool_calls.append({
                "id": (
                    getattr(call, "id", None)
                    or str(uuid.uuid4())
                ),
                "name": call.name,
                "arguments": dict(
                    call.args or {}
                ),
            })

        # Usage field names vary slightly by SDK release.
        usage_metadata = getattr(
            response,
            "usage_metadata",
            None
        )
        if usage_metadata is None:
            usage_dict = {}
        elif hasattr(usage_metadata, "model_dump"):
            usage_dict = usage_metadata.model_dump()
        else:
            usage_dict = {
                "raw": str(usage_metadata)
            }

        text = ""
        try:
            text = response.text or ""
        except Exception:
            text = ""

        # Preserve the exact Gemini model Content object. This can contain
        # the required thought_signature on functionCall parts.
        gemini_content = None
        try:
            if (
                response.candidates
                and response.candidates[0].content is not None
            ):
                gemini_content = response.candidates[0].content
        except Exception:
            gemini_content = None

        return {
            "text": self._clean_response(text),
            "tool_calls": tool_calls,
            "provider": "gemini",
            "model": Config.GEMINI_MODEL,
            "usage": usage_dict,
            "gemini_content": gemini_content,
        }

    def generate_turn(
        self,
        messages: list,
        tools: list,
        provider: str,
        tool_choice: str = "auto",
        max_new_tokens: int = 512,
    ) -> dict:
        start = time.time()
        self.total_api_attempts += 1

        try:
            if provider == "gemini":
                result = self._gemini_turn(
                    messages,
                    tools,
                    tool_choice,
                    max_new_tokens,
                )
            elif provider == "qwen":
                result = self._qwen_turn(
                    messages,
                    tools,
                    tool_choice,
                    max_new_tokens,
                )
            else:
                raise ValueError(
                    f"Unknown provider: {provider}"
                )

            elapsed_ms = int(
                (time.time() - start) * 1000
            )
            self.total_llm_calls += 1
            self.total_llm_time += elapsed_ms

            self.last_provider_used = result["provider"]
            self.last_model_used = result["model"]
            self.last_usage = result.get(
                "usage",
                {}
            )
            self.last_error = None

            return result

        except Exception as exc:
            self.last_error = (
                f"{provider}: "
                f"{type(exc).__name__}: {exc}"
            )
            logger.exception(
                "%s model request failed: %s",
                provider,
                exc,
            )
            raise

    def generate_direct(
        self,
        prompt: str,
        model_mode: str = "Auto",
        max_new_tokens: int = 512,
    ) -> str:
        provider = self.choose_provider(
            model_mode
        )

        messages = [{
            "role": "user",
            "content": prompt,
        }]

        try:
            result = self.generate_turn(
                messages=messages,
                tools=RAILWAY_TOOLS,
                provider=provider,
                tool_choice="none",
                max_new_tokens=max_new_tokens,
            )
            return result["text"]

        except Exception:
            fallback = self.fallback_provider(
                provider,
                model_mode
            )
            if fallback:
                result = self.generate_turn(
                    messages=messages,
                    tools=RAILWAY_TOOLS,
                    provider=fallback,
                    tool_choice="none",
                    max_new_tokens=max_new_tokens,
                )
                return result["text"]
            raise

    def get_stats(self) -> dict:
        return {
            "total_calls": self.total_llm_calls,
            "api_attempts": self.total_api_attempts,
            "avg_time_ms": int(
                self.total_llm_time
                / max(1, self.total_llm_calls)
            ),
            "total_time_ms": self.total_llm_time,
            "last_provider_used": self.last_provider_used,
            "last_model_used": self.last_model_used,
            "last_error": self.last_error,
            "last_usage": self.last_usage,
            "gemini_available": self.gemini_available,
            "gemini_model": Config.GEMINI_MODEL,
            "vertex_project": self.vertex_project_id,
            "vertex_location": Config.GEMINI_LOCATION,
            "qwen_available": self.qwen_available,
            "qwen_model": Config.QWEN_MODEL,
            "default_mode": Config.DEFAULT_MODEL_MODE,
        }


# Backward-compatible name for old cells.
LLMService = HybridLLMService


In [ ]:
# ============================================
# MEMORY MONITOR
# ============================================

import torch


class MemoryMonitor:
    """
    Monitor local runtime memory.

    The LLM now runs remotely, so GPU memory should remain near zero
    unless another part of the notebook explicitly uses CUDA.
    """

    def __init__(self):
        self.peak_allocated = 0
        self.peak_reserved = 0

    def check(self, label=""):
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3

            self.peak_allocated = max(
                self.peak_allocated,
                allocated
            )
            self.peak_reserved = max(
                self.peak_reserved,
                reserved
            )

            print(
                f"💾 {label}: "
                f"{allocated:.2f}GB / {reserved:.2f}GB "
                f"(Peak: {self.peak_allocated:.2f}GB)"
            )

            return allocated, reserved

        print(f"💾 {label}: CPU runtime (no CUDA allocation)")
        return 0, 0

    def get_summary(self):
        if torch.cuda.is_available():
            return {
                "peak_allocated": f"{self.peak_allocated:.2f}GB",
                "peak_reserved": f"{self.peak_reserved:.2f}GB",
                "current_allocated": (
                    f"{torch.cuda.memory_allocated() / 1024**3:.2f}GB"
                ),
                "current_reserved": (
                    f"{torch.cuda.memory_reserved() / 1024**3:.2f}GB"
                ),
            }

        return {
            "status": "CPU/local LLM disabled"
        }


In [ ]:
# ============================================================
# RAILWAY API CLIENT
# ============================================================

import requests
import logging
import math
from typing import Optional, Dict, Any
from urllib.parse import quote

logger = logging.getLogger(__name__)


class RailwayAPIClient:
    """
    Safe client for passenger-facing Railway APIs.

    IMPORTANT:
    Expose only read-only passenger APIs plus non-mutating fare calculation.
    Booking mutation endpoints are intentionally not implemented here.
    """

    def __init__(
        self,
        base_url: str = "http://127.0.0.1:8000",
        timeout: int = 10
    ):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout

    # --------------------------------------------------------
    # Generic GET
    # --------------------------------------------------------

    def _get(
        self,
        endpoint: str,
        params: Optional[Dict[str, Any]] = None
    ) -> Dict[str, Any]:

        url = f"{self.base_url}{endpoint}"

        try:
            logger.info(f"🚂 Railway API: GET {url}")

            response = requests.get(
                url,
                params=params or {},
                timeout=self.timeout
            )

            response.raise_for_status()

            data = response.json()

            return {
                "success": True,
                "data": data
            }

        except requests.exceptions.ConnectionError as e:

            logger.error(f"❌ Railway API connection failed: {e}")

            return {
                "success": False,
                "error": "Railway backend is unavailable.",
                "details": str(e)
            }

        except requests.exceptions.Timeout as e:

            logger.error(f"❌ Railway API timeout: {e}")

            return {
                "success": False,
                "error": "Railway backend request timed out.",
                "details": str(e)
            }

        except requests.exceptions.HTTPError as e:

            logger.error(f"❌ Railway API HTTP error: {e}")

            try:
                detail = response.json()
            except Exception:
                detail = response.text

            return {
                "success": False,
                "error": "Railway API returned an error.",
                "status_code": response.status_code,
                "details": detail
            }

        except Exception as e:

            logger.error(
                f"❌ Unexpected Railway API error: {e}",
                exc_info=True
            )

            return {
                "success": False,
                "error": "Unexpected Railway API error.",
                "details": str(e)
            }

    # --------------------------------------------------------
    # Generic non-mutating POST (used only for fare calculation)
    # --------------------------------------------------------

    def _post_json(
        self,
        endpoint: str,
        body: Dict[str, Any]
    ) -> Dict[str, Any]:

        url = f"{self.base_url}{endpoint}"

        try:
            logger.info(f"🚂 Railway API: POST {url}")

            response = requests.post(
                url,
                json=body,
                timeout=self.timeout
            )

            response.raise_for_status()

            data = response.json()

            return {
                "success": True,
                "data": data
            }

        except requests.exceptions.ConnectionError as e:
            logger.error(f"❌ Railway API connection failed: {e}")
            return {
                "success": False,
                "error": "Railway backend is unavailable.",
                "details": str(e)
            }

        except requests.exceptions.Timeout as e:
            logger.error(f"❌ Railway API timeout: {e}")
            return {
                "success": False,
                "error": "Railway backend request timed out.",
                "details": str(e)
            }

        except requests.exceptions.HTTPError as e:
            logger.error(f"❌ Railway API HTTP error: {e}")

            try:
                detail = response.json()
            except Exception:
                detail = response.text

            return {
                "success": False,
                "error": "Railway API returned an error.",
                "status_code": response.status_code,
                "details": detail
            }

        except Exception as e:
            logger.error(
                f"❌ Unexpected Railway API error: {e}",
                exc_info=True
            )
            return {
                "success": False,
                "error": "Unexpected Railway API error.",
                "details": str(e)
            }

    # ========================================================
    # PUBLIC PASSENGER APIs
    # ========================================================

    def search_stations(
        self,
        query: str,
        limit: int = 10
    ) -> Dict[str, Any]:

        # Prevent:
        # /api/stations/search?q=
        if not query or not query.strip():
            return {
                "success": False,
                "error": "Station search requires a station name or code."
            }

        query = query.strip()

        limit = max(1, min(int(limit), 20))

        result = self._get(
            "/api/stations/search",
            params={
                "q": query,
                "limit": limit
            }
        )

        return result

    # --------------------------------------------------------
    # Get Station
    # --------------------------------------------------------

    def get_station(
        self,
        station_id: int
    ) -> Dict[str, Any]:

        return self._get(
            f"/api/stations/{station_id}"
        )
    # --------------------------------------------------------
    # Train response normalization
    # --------------------------------------------------------
    @staticmethod
    def _normalize_train_record(train: Dict[str, Any]) -> Dict[str, Any]:
        """
        Convert a backend train object into the exact Train model
        structure that should be shown to the AI.

        IMPORTANT:
        - Do not invent values.
        - Do not translate values.
        - Keep database values exactly as returned by FastAPI.
        """

        if not isinstance(train, dict):
            return {}

        return {
            "id": train.get("id"),
            "train_no": train.get("train_no"),
            "train_name": train.get("train_name"),
            "train_type": train.get("train_type"),
            "route_id": train.get("route_id"),
            "total_coaches": train.get("total_coaches"),
            "capacity": train.get("capacity"),
            "speed": train.get("speed"),
            "status": train.get("status"),
            "created_at": train.get("created_at"),
            "updated_at": train.get("updated_at"),
        }


    @staticmethod
    def _unwrap_single_train(payload):
        """
        Handle common FastAPI response shapes:

        {...train fields...}

        {"data": {...train fields...}}

        {"train": {...train fields...}}
        """

        if not isinstance(payload, dict):
            return payload

        if isinstance(payload.get("train"), dict):
            return payload["train"]

        if isinstance(payload.get("data"), dict):
            return payload["data"]

        return payload


    @staticmethod
    def _unwrap_train_list(payload):
        """
        Handle common train-list response shapes:

        [...]
        {"data": [...]}
        {"trains": [...]}
        {"items": [...]}
        """

        if isinstance(payload, list):
            return payload

        if not isinstance(payload, dict):
            return []

        for key in ("trains", "items", "data"):
            value = payload.get(key)

            if isinstance(value, list):
                return value

        return []
    # --------------------------------------------------------
    # List Trains
    # --------------------------------------------------------

    def list_trains(
        self,
        status: Optional[str] = "ACTIVE",
        limit: int = 50
    ) -> Dict[str, Any]:

        params = {
            "limit": max(1, min(int(limit), 100))
        }

        if status:
            params["status"] = str(status).strip().upper()

        result = self._get(
            "/api/trains/catalog",
            params=params
        )

        if not result.get("success"):
            return result

        raw_payload = result.get("data")
        train_list = self._unwrap_train_list(raw_payload)

        normalized_trains = [
            self._normalize_train_record(train)
            for train in train_list
            if isinstance(train, dict)
        ]

        return {
            "success": True,
            "data": normalized_trains,
            "count": len(normalized_trains)
        }

    # --------------------------------------------------------
    # Search Trains / Resolve passenger-visible train -> train_id
    # --------------------------------------------------------

    def search_trains(
        self,
        query: str,
        status: Optional[str] = "ACTIVE",
        limit: int = 10
    ) -> Dict[str, Any]:

        if not query or not str(query).strip():
            return {
                "success": False,
                "error": "Train search requires a train number or train name."
            }

        params = {
            "q": str(query).strip(),
            "limit": max(1, min(int(limit), 20))
        }

        if status:
            params["status"] = str(status).strip().upper()

        result = self._get(
            "/api/trains/search",
            params=params
        )

        if not result.get("success"):
            return result

        raw_payload = result.get("data")
        train_list = self._unwrap_train_list(raw_payload)

        normalized_trains = [
            self._normalize_train_record(train)
            for train in train_list
            if isinstance(train, dict)
        ]

        return {
            "success": True,
            "query": str(query).strip(),
            "data": normalized_trains,
            "count": len(normalized_trains)
        }

    # --------------------------------------------------------
    # Get Train
    # --------------------------------------------------------
    def get_train(
        self,
        train_id: int
    ) -> Dict[str, Any]:

        result = self._get(
            f"/api/trains/{train_id}"
        )

        if not result.get("success"):
            return result

        raw_payload = result.get("data")

        train = self._unwrap_single_train(
            raw_payload
        )

        normalized_train = self._normalize_train_record(
            train
        )

        if not normalized_train.get("id"):
            return {
                "success": False,
                "error": "Train data returned by backend is invalid.",
                "raw_data": raw_payload
            }

        return {
            "success": True,
            "data": normalized_train
        }

    # --------------------------------------------------------
    # Get Train Stops
    # --------------------------------------------------------

    def get_train_stops(
        self,
        train_id: int
    ) -> Dict[str, Any]:

        return self._get(
            f"/api/train-stops/train/{train_id}"
        )

    # --------------------------------------------------------
    # Get Route
    # --------------------------------------------------------

    def get_route(
        self,
        route_id: int
    ) -> Dict[str, Any]:

        return self._get(
            f"/api/routes/{route_id}"
        )

    # --------------------------------------------------------
    # Get Trains By Route
    # --------------------------------------------------------
    def get_trains_by_route(
        self,
        route_id: int,
        status: Optional[str] = None
    ) -> Dict[str, Any]:

        params = {}

        if status:
            # Database currently stores values such as ACTIVE.
            params["status"] = str(status).strip().upper()

        result = self._get(
            f"/api/trains/by-route/{route_id}",
            params=params
        )

        if not result.get("success"):
            return result

        raw_payload = result.get("data")

        train_list = self._unwrap_train_list(
            raw_payload
        )

        normalized_trains = [
            self._normalize_train_record(train)
            for train in train_list
            if isinstance(train, dict)
        ]

        return {
            "success": True,
            "data": normalized_trains,
            "count": len(normalized_trains),
            "route_id": route_id
        }


    # --------------------------------------------------------
    # Search Dynamic Schedules Between Stations
    # --------------------------------------------------------
    def search_schedules(
        self,
        from_station_id: int,
        to_station_id: int,
        route_ids: list,
        date_from: str,
        date_to: str,
    ) -> Dict[str, Any]:

        route_ids = [int(rid) for rid in route_ids]

        return self._get(
            "/api/schedules/search",
            params={
                "from_station_id": int(from_station_id),
                "to_station_id": int(to_station_id),
                "route_ids": ",".join(str(rid) for rid in route_ids),
                "date_from": str(date_from),
                "date_to": str(date_to),
            }
        )

    # --------------------------------------------------------
    # Get Route + Train Schedule
    # --------------------------------------------------------

    def get_route_train_schedule(
        self,
        route_id: int,
        train_id: int
    ) -> Dict[str, Any]:

        return self._get(
            f"/api/routes/{route_id}/schedule/{train_id}"
        )

    # --------------------------------------------------------
    # Get Next Train
    # --------------------------------------------------------

    def get_next_train(
        self,
        route_id: int,
        train_id: int,
        station_order: int
    ) -> Dict[str, Any]:

        return self._get(
            f"/api/routes/{route_id}/next-train/{train_id}",
            params={"station_order": int(station_order)}
        )

    # --------------------------------------------------------
    # Exact configured fare
    # --------------------------------------------------------

    @staticmethod
    def _normalize_fare_class(
        class_type: Optional[str]
    ) -> str:

        raw = str(
            class_type or "ECONOMY_CLASS"
        ).strip().upper()

        aliases = {
            "ECONOMY": "ECONOMY_CLASS",
            "ECONOMY CLASS": "ECONOMY_CLASS",
            "ECONOMY_CLASS": "ECONOMY_CLASS",
            "ORDINARY": "ECONOMY_CLASS",
            "ORDINARY CLASS": "ECONOMY_CLASS",
            "ORDINARY_CLASS": "ECONOMY_CLASS",
            "UPPER": "UPPER_CLASS",
            "UPPER CLASS": "UPPER_CLASS",
            "UPPER_CLASS": "UPPER_CLASS",
            "SLEEPER": "SLEEPER",
            "SLEEPER CLASS": "SLEEPER",
            "SLEEPER_CLASS": "SLEEPER",
        }

        normalized = aliases.get(raw, raw)

        if normalized not in {
            "ECONOMY_CLASS",
            "UPPER_CLASS",
            "SLEEPER",
        }:
            raise ValueError(
                "Current backend fare classes are "
                "ECONOMY_CLASS, UPPER_CLASS, SLEEPER."
            )

        return normalized

    def calculate_fare(
        self,
        train_id: int,
        from_route_station_id: int,
        to_route_station_id: int,
        class_type: str = "ECONOMY_CLASS",
        seat_type: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        Backend FeeCalculationRequest unfortunately names its RouteStation
        identifiers from_station_id/to_station_id. Keep the public chatbot
        interface explicit so public Station IDs cannot be confused with
        RouteStation row IDs.
        """

        body = {
            "train_id": int(train_id),
            "from_station_id": int(from_route_station_id),
            "to_station_id": int(to_route_station_id),
            "class_type": self._normalize_fare_class(class_type),
            "seat_type": (
                str(seat_type).strip()
                if seat_type is not None and str(seat_type).strip()
                else None
            ),
        }

        return self._post_json(
            "/api/fees/calculate",
            body=body
        )

    # --------------------------------------------------------
    # Fare matrix
    # --------------------------------------------------------

    def get_fare_price_matrix(
        self,
        train_id: int,
        class_type: str = "ECONOMY_CLASS"
    ) -> Dict[str, Any]:

        return self._get(
            f"/api/fees/price-matrix/{int(train_id)}",
            params={
                "class_type": self._normalize_fare_class(
                    class_type
                )
            }
        )


    # --------------------------------------------------------
    # Yangon-Pyay small-consignment parcel fee
    # --------------------------------------------------------

    @staticmethod
    def _looks_like_yangon_pyay_route(route: Dict[str, Any]) -> bool:
        combined = " ".join(
            str(route.get(key) or "")
            for key in ("name", "origin", "destination")
        ).casefold()
        return (
            ("yangon" in combined or "ရန်ကုန်" in combined)
            and ("pyay" in combined or "ပြည်" in combined)
        )

    @staticmethod
    def _round_parcel_fee_to_50(raw_fee_kyat: float) -> int:
        return int(math.ceil(float(raw_fee_kyat) / 50.0) * 50)

    def calculate_parcel_fee(
        self,
        route_id: int,
        from_station_id: int,
        to_station_id: int,
        weight_viss: float,
    ) -> Dict[str, Any]:
        """
        Small-consignment parcel fee for Yangon-Pyay/Pyay-Yangon.

        Official flat-route rule:
        13 pyas = 0.13 MMK / viss / mile
        minimum weight = 5 viss
        minimum distance = 100 miles
        final fee rounded upward to a 50-MMK boundary
        """
        try:
            route_id = int(route_id)
            from_station_id = int(from_station_id)
            to_station_id = int(to_station_id)
            weight_viss = float(weight_viss)
        except (TypeError, ValueError):
            return {"success": False, "error": "Invalid parcel calculation arguments."}

        if min(route_id, from_station_id, to_station_id) <= 0 or weight_viss <= 0:
            return {"success": False, "error": "Parcel calculation arguments must be positive."}

        route_result = self.get_route(route_id)
        if not route_result.get("success"):
            return route_result

        route = route_result.get("data") or {}
        if not self._looks_like_yangon_pyay_route(route):
            return {
                "success": False,
                "error": "Parcel calculation is supported only for Yangon-Pyay/Pyay-Yangon."
            }

        stations = route.get("stations") or []

        def find_station(station_id):
            for item in stations:
                try:
                    if item.get("station_id") is not None and int(item["station_id"]) == station_id:
                        return item
                except (TypeError, ValueError):
                    pass
            return None

        origin = find_station(from_station_id)
        destination = find_station(to_station_id)

        if not origin or not destination:
            return {
                "success": False,
                "error": "Both stations must belong to the selected Yangon-Pyay/Pyay-Yangon route."
            }

        a = origin.get("distance_from_origin")
        b = destination.get("distance_from_origin")
        if a is None or b is None:
            return {"success": False, "error": "distance_from_origin is missing for a selected station."}

        try:
            actual_distance_miles = abs(float(b) - float(a))
        except (TypeError, ValueError):
            return {"success": False, "error": "Route-station distance is not numeric."}

        if actual_distance_miles <= 0:
            return {"success": False, "error": "Origin and destination must be different stations."}

        rate_pyas = 13.0
        rate_mmk = 0.13
        minimum_weight = 5.0
        minimum_distance = 100.0

        chargeable_weight = max(weight_viss, minimum_weight)
        chargeable_distance = max(actual_distance_miles, minimum_distance)

        raw_fee = rate_mmk * chargeable_weight * chargeable_distance
        final_fee = self._round_parcel_fee_to_50(raw_fee)

        return {
            "success": True,
            "data": {
                "calculation_type": "SMALL_CONSIGNMENT_FLAT_ROUTE",
                "route_name": route.get("name"),
                "from_station": origin.get("station_name"),
                "to_station": destination.get("station_name"),
                "actual_distance_miles": round(actual_distance_miles, 3),
                "minimum_distance_miles": minimum_distance,
                "chargeable_distance_miles": round(chargeable_distance, 3),
                "actual_weight_viss": round(weight_viss, 3),
                "minimum_weight_viss": minimum_weight,
                "chargeable_weight_viss": round(chargeable_weight, 3),
                "rate_pyas_per_viss_mile": rate_pyas,
                "rate_mmk_per_viss_mile": rate_mmk,
                "raw_fee_kyat": round(raw_fee, 2),
                "final_fee_kyat": final_fee,
                "rounding_rule": "Round upward to the applicable 50-MMK boundary.",
                "official_source": "https://www.railways.gov.mm/parcel-rate/"
            }
        }

    # --------------------------------------------------------
    # Schedule-specific seat map
    # --------------------------------------------------------

    def get_schedule_seat_map(
        self,
        schedule_id: int
    ) -> Dict[str, Any]:

        return self._get(
            f"/api/seats/schedule/{int(schedule_id)}"
        )

    # --------------------------------------------------------
    # Exact seat availability on a schedule
    # --------------------------------------------------------

    def check_seat_availability(
        self,
        seat_id: int,
        schedule_id: int
    ) -> Dict[str, Any]:

        return self._get(
            f"/api/seats/{int(seat_id)}/availability",
            params={"schedule_id": int(schedule_id)}
        )

    # --------------------------------------------------------
    # Read-only ticket journey status
    # --------------------------------------------------------

    def get_ticket_journey_status(
        self,
        ticket_no: str
    ) -> Dict[str, Any]:
        """
        Use only the backend's passenger-facing journey-status endpoint.

        Do NOT call the full booking-by-ticket endpoint here because that
        response contains passenger identity/contact fields.
        """

        ticket_no = str(ticket_no or "").strip()

        if not ticket_no:
            return {
                "success": False,
                "error": "A ticket number is required."
            }

        result = self._get(
            "/api/bookings/ticket/"
            + quote(ticket_no, safe="")
            + "/journey-status"
        )

        if not result.get("success"):
            return result

        payload = result.get("data")
        if not isinstance(payload, dict):
            return {
                "success": False,
                "error": "Ticket status returned by backend is invalid."
            }

        # Defense in depth: keep only the passenger-facing fields from the
        # journey-status contract. Unknown future fields are not passed to LLM.
        allowed = {
            "ticket_no",
            "booking_no",
            "booking_status",
            "schedule_id",
            "schedule_status",
            "train_id",
            "travel_date",
            "boarding_station",
            "destination_station",
            "last_reached",
            "stops",
        }

        sanitized = {
            key: value
            for key, value in payload.items()
            if key in allowed
        }

        return {
            "success": True,
            "data": sanitized
        }

    # --------------------------------------------------------
    # Health
    # --------------------------------------------------------

    def health_check(self) -> Dict[str, Any]:

        return self._get("/health")


In [ ]:
# ============================================================
# TRAIN API DISCOVERY TESTS
# ============================================================
# These two endpoints must exist on the deployed FastAPI backend for
# RailMM to resolve passenger-visible train names/numbers into train_id.

import json

api = RailwayAPIClient(
    base_url=Config.RAILWAY_API_BASE_URL,
    timeout=Config.RAILWAY_API_TIMEOUT
)

print("\n1) Train catalog")
catalog_test = api.list_trains(status="ACTIVE", limit=10)
print(json.dumps(catalog_test, ensure_ascii=False, indent=2, default=str))

print("\n2) Train search / entity resolution")
search_test = api.search_trains("72", status="ACTIVE", limit=10)
print(json.dumps(search_test, ensure_ascii=False, indent=2, default=str))

print("\n3) Existing specific-train lookup")
train_test = api.get_train(6)
print(json.dumps(train_test, ensure_ascii=False, indent=2, default=str))

if not catalog_test.get("success") or not search_test.get("success"):
    print(
        "\n⚠️ The chatbot routing code is ready, but the deployed backend still "
        "needs GET /api/trains/catalog and GET /api/trains/search?q=... "
        "before train discovery can work end-to-end."
    )



1) Train catalog
{
  "success": true,
  "data": [
    {
      "id": 6,
      "train_no": "73",
      "train_name": "အမှတ်(၇၃)အဆန်",
      "train_type": "EXPRESS",
      "route_id": 5,
      "total_coaches": 6,
      "capacity": 290,
      "speed": 41.63186987990137,
      "status": "ACTIVE",
      "created_at": "2026-08-22T15:20:16.555568+00:00",
      "updated_at": "2026-08-22T15:20:16.555568+00:00"
    },
    {
      "id": 8,
      "train_no": "T-321",
      "train_name": "မြို့ပတ်အမြန်ရထား",
      "train_type": "LOCAL",
      "route_id": 13,
      "total_coaches": 5,
      "capacity": 141,
      "speed": 24.0,
      "status": "ACTIVE",
      "created_at": "2026-08-22T15:20:16.555568+00:00",
      "updated_at": "2026-09-01T08:10:33.375871+00:00"
    },
    {
      "id": 9,
      "train_no": "74",
      "train_name": "အမှတ်(၇၄)အစုန်",
      "train_type": "EXPRESS",
      "route_id": 14,
      "total_coaches": 6,
      "capacity": 290,
      "speed": 50.0,
      "status": "ACTIVE",
  

## Cloud Run and CORS

The chatbot notebook calls FastAPI from Python (`requests`) running in Colab.
That is **server-to-server traffic**, so browser CORS does not block the
notebook.

Use this production backend:

```text
https://smart-railway-platform-git-803829337765.asia-southeast1.run.app
```

CORS matters when the **React app in a user's browser** calls Cloud Run
directly. Keep the FastAPI `allow_origins` list restricted to the actual
frontend origins, for example:

```python
allow_origins=[
    "http://localhost:5173",
    "https://your-production-frontend.example",
]
```

Do not add Colab or Gradio URLs to CORS merely for this notebook.

If the Cloud Run service itself is IAM-private, that is a different issue from
CORS: the notebook would need Cloud Run authentication. A normal public
FastAPI service with restricted browser CORS does not require that.


In [ ]:
# ============================================================
# TEST DEPLOYED RAILWAY API CONNECTION
# ============================================================

api = RailwayAPIClient(
    base_url=Config.RAILWAY_API_BASE_URL,
    timeout=Config.RAILWAY_API_TIMEOUT
)

print("Testing deployed Railway backend...")
print("Base URL:", Config.RAILWAY_API_BASE_URL)

if "smart-railway-api.onrender.com" in Config.RAILWAY_API_BASE_URL.casefold():
    raise RuntimeError(
        "The retired Render backend is still configured. "
        "Remove/update the RAILWAY_API_BASE_URL Colab Secret or environment "
        "variable and use the Google Cloud Run backend."
    )

if not Config.RAILWAY_API_BASE_URL.startswith("https://"):
    raise RuntimeError(
        "RAILWAY_API_BASE_URL must use HTTPS for the deployed backend."
    )

health = api.health_check()
print("\nHealth:")
print(health)

if not health.get("success"):
    raise RuntimeError(
        "Colab cannot reach the Google Cloud Run backend. "
        "Check the Cloud Run URL and make sure the required read-only "
        "railway API endpoints are reachable from server-to-server clients. "
        "Browser CORS settings are not the cause of a Colab requests failure."
    )

print("\nTesting station search...")
station_test = api.search_stations("Yangon", limit=5)
print(station_test)

if station_test.get("success"):
    print("\n✅ Colab -> deployed FastAPI connection works.")
else:
    print("\n⚠️ Backend is reachable, but /api/stations/search returned an error.")


Testing deployed Railway backend...
Base URL: https://smart-railway-platform-git-803829337765.asia-southeast1.run.app

Health:
{'success': True, 'data': {'status': 'healthy', 'message': 'API is running', 'version': '2.1.0', 'chatbot_enabled': True, 'colab_connected': True, 'colab_url': 'https://6ce7117cb1e441c923.gradio.live'}}

Testing station search...
{'success': True, 'data': []}

✅ Colab -> deployed FastAPI connection works.


In [ ]:
# ============================================
# RAG ENGINE - DIRECT MULTILINGUAL QWEN
# ============================================

from typing import Optional, List, Dict, Any, Tuple
import gc
import time
import json
import hashlib
import re


def contains_myanmar(text: str) -> bool:
    """Return True when text contains Myanmar Unicode characters."""
    if not text:
        return False

    return bool(
        re.search(
            r"[\u1000-\u109F\uA9E0-\uA9FF\uAA60-\uAA7F]",
            text
        )
    )



_DYNAMIC_RAILWAY_HINTS = (
    # Myanmar: schedules / times / changing service
    "ဘယ်ချိန်", "အချိန်", "မနက်ဖြန်", "ဒီနေ့", "ယနေ့",
    "နောက်ရထား", "ရောက်မယ့်", "ထွက်မယ့်", "နှောင့်နှေး",

    # Myanmar: train-specific stops
    "ရပ်လား", "ရပ်မလား", "ရပ်သလား", "ရပ်ပါသလား",
    "ရပ်နား", "ရပ်တဲ့ဘူတာ", "ဘယ်ဘူတာတွေမှာရပ်",

    # Myanmar: exact fares
    "လက်မှတ်ခ", "ရထားခ", "ခဘယ်လောက်", "ဘယ်လောက်ကျ",

    # Myanmar: seats / availability
    "ခုံလွတ်", "ထိုင်ခုံလွတ်", "ထိုင်ခုံရနိုင်", "ခုံရနိုင်",
    "ထိုင်ခုံ", "seat",

    # English: schedules / times
    "schedule", "departure", "arrival", "today", "tomorrow",
    "next train", "delay",

    # English: train-specific stops
    "stop at", "stops at", "train stops", "stopping station",
    "stopping stations", "which stations does", "where does train",

    # English: exact fare
    "fare", "ticket price", "train fare", "how much is the ticket",

    # English: seats / availability
    "available seat", "available seats", "seat availability",
    "seat available", "seats available",

    # Myanmar/English: exact parcel fee calculation
    "ပါဆယ်ခ", "ပါဆယ်တန်ဆာခ", "ကုန်ပို့ခ",
    "တန်ဆာခ ဘယ်လောက်", "parcel fee", "parcel cost",
    "parcel charge", "shipping cost",

    # Ticket / booking journey status (specific phrases only; generic
    # ticketing rules still belong to RAG)
    "လက်မှတ်အခြေအနေ", "ဘွတ်ကင်အခြေအနေ", "ခရီးစဉ်အခြေအနေ",
    "လက်မှတ်နံပါတ်", "booking status", "ticket status",
    "journey status", "track my ticket",
)


def requires_dynamic_railway_api(question: str) -> bool:
    """
    Return True for facts that must be backed by structured/backend data.

    Keep this deliberately narrower than generic words such as "ticket":
    ORTP rules, refunds, passenger law, and ticketing explanations are valid
    RAG questions and should not be forced into a live-data path.
    """
    q = str(question or "").casefold()
    return any(hint.casefold() in q for hint in _DYNAMIC_RAILWAY_HINTS)


def operational_data_unavailable_message(question: str) -> str:
    """Controlled response when an exact operational fact cannot be retrieved."""
    if contains_myanmar(question):
        return (
            "တောင်းပန်ပါတယ်။ ဒီမေးခွန်းအတွက် လိုအပ်တဲ့ လက်ရှိ "
            "ရထားဝန်ဆောင်မှုအချက်အလက်ကို backend မှ မရယူနိုင်သေးပါဘူး။ "
            "အချိန်ဇယား၊ ရပ်နားဘူတာ၊ လက်မှတ်ခ၊ ထိုင်ခုံလက်ကျန် သို့မဟုတ် ဘွတ်ကင်အခြေအနေကို "
            "မခန့်မှန်းဘဲ အတည်ပြုနိုင်သည့်အချိန်မှသာ ဖြေကြားပေးပါမယ်။"
        )

    return (
        "Sorry, I can't retrieve the required operational railway data right now. "
        "I won't guess schedules, train stops, fares, parcel charges, seat availability, or booking status."
    )


def is_safe_operational_clarification(text: str) -> bool:
    """
    Allow a short question that asks for a genuinely missing passenger detail
    (date, train, class, ticket number, seat, etc.) without treating it as an
    ungrounded operational answer.
    """
    value = str(text or "").strip()

    if not value or len(value) > 500:
        return False

    lowered = value.casefold()

    question_signals = (
        "?",
        "which train",
        "what date",
        "which date",
        "travel date",
        "which class",
        "ticket number",
        "seat number",
        "ဘယ်ရထား",
        "ဘယ်နေ့",
        "ရက်စွဲ",
        "အတန်း",
        "လက်မှတ်နံပါတ်",
        "ခုံနံပါတ်",
        "ပြောပြ",
        "ရွေး",
    )

    # A clarification must look like a request for missing input, not a
    # passenger-facing claim containing an operational value.
    value_claim_signals = (
        " kyat",
        "ကျပ်",
        "available seats:",
        "fare is",
        "departs at",
        "arrives at",
    )

    return (
        any(signal.casefold() in lowered for signal in question_signals)
        and not any(
            signal.casefold() in lowered
            for signal in value_claim_signals
        )
    )


class RAGEngine:
    """
    Singleton RAG Engine with lazy loading.

    Normal path:
        User Myanmar/English question
            -> BGE-M3 direct multilingual embedding
            -> MongoDB Atlas vector search
            -> Gemini primary / Qwen fallback with native railway tools
            -> final answer in user's language

    Fallback path (Myanmar only, weak retrieval):
        original Myanmar query
            -> translation to English
            -> second BGE-M3 search
            -> Qwen still answers DIRECTLY in Myanmar

    There is no final-answer translation layer.
    """

    _instance = None
    _initialized = False

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        if RAGEngine._initialized:
            print("✅ RAG Engine already initialized, reusing...")
            return

        print("=" * 60)
        print("Initializing Railway AI...")
        print("=" * 60)

        self._embedding = None
        self._mongo = None
        self._translator = None
        self._llm = None
        self._prompt_builder = None
        self._railway_api = None
        self._railway_catalog = None

        self.response_cache = {}
        self.memory_monitor = MemoryMonitor()

        RAGEngine._initialized = True

        print("✅ RAG Engine initialized")
        print("✅ LLM routing: Gemini primary -> Qwen fallback (Auto mode)")
        print("✅ Direct multilingual RAG: enabled")
        print("✅ Translation: retrieval fallback only")
        print("=" * 60)

    @property
    def railway_api(self):
        if self._railway_api is None:
            self._railway_api = RailwayAPIClient(
                base_url=Config.RAILWAY_API_BASE_URL,
                timeout=Config.RAILWAY_API_TIMEOUT
            )
            logger.info(
                "🚂 Railway API client initialized: %s",
                Config.RAILWAY_API_BASE_URL
            )
        return self._railway_api

    @property
    def embedding(self):
        if self._embedding is None:
            print("🔄 Loading BGE-M3 on CPU...")
            self._embedding = EmbeddingService()
        return self._embedding

    @property
    def mongo(self):
        if self._mongo is None:
            print("🔄 Loading MongoDB Service...")
            self._mongo = MongoService()
        return self._mongo

    @property
    def translator(self):
        """
        Lazy load ONLY when direct Myanmar retrieval is weak.
        """
        if self._translator is None:
            print("🔄 Loading retrieval translation fallback...")
            self._translator = DeepTranslationService()
        return self._translator

    @property
    def llm(self):
        if self._llm is None:
            print("🔄 Initializing Gemini/Qwen hybrid LLM service...")
            self._llm = HybridLLMService(
                stable_knowledge=self.railway_catalog.system_knowledge
            )
        return self._llm

    @property
    def railway_catalog(self):
        if self._railway_catalog is None:
            # Reuse the catalog loaded earlier, or load it lazily.
            self._railway_catalog = globals().get("railway_catalog")
            if self._railway_catalog is None:
                self._railway_catalog = RailwayReferenceCatalog(
                    Config.RAILWAY_CATALOG_PATH
                )
        return self._railway_catalog

    @property
    def prompt_builder(self):
        if self._prompt_builder is None:
            self._prompt_builder = PromptBuilder(
                catalog=self.railway_catalog
            )
        return self._prompt_builder

    @classmethod
    def get_instance(cls):
        if cls._instance is None:
            cls._instance = RAGEngine()
        return cls._instance

    @classmethod
    def reset(cls):
        if cls._instance is not None:
            print("🔄 Resetting RAG Engine...")

            try:
                cls._instance.__del__()
            except Exception:
                pass

            cls._instance = None
            cls._initialized = False

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            gc.collect()
            print("🧹 RAG Engine reset complete")

    def _cache_key(
        self,
        question: str,
        history: List[Dict],
        model_mode: str = "Auto"
    ) -> str:
        """
        Include recent conversation context in the cache key so the same
        question in different conversations does not reuse a wrong answer.
        """
        recent_history = history[-6:] if history else []

        raw = json.dumps(
            {
                "question": question,
                "history": recent_history,
                "model_mode": HybridLLMService.normalize_model_mode(
                    model_mode
                ),
            },
            ensure_ascii=False,
            sort_keys=True,
            default=str,
        )

        return hashlib.sha256(
            raw.encode("utf-8")
        ).hexdigest()

    def _retrieve(
        self,
        query: str,
        min_score: float
    ) -> Tuple[List[Dict], float, List[Dict]]:
        """
        Return:
            filtered_docs,
            top_raw_score,
            raw_candidates
        """
        query_embedding = self.embedding.encode(query)

        candidates = self.mongo.vector_search(
            query_embedding
        )

        candidates = sorted(
            candidates,
            key=lambda d: float(d.get("score", 0)),
            reverse=True
        )

        top_score = (
            float(candidates[0].get("score", 0))
            if candidates
            else 0.0
        )

        docs = [
            d
            for d in candidates
            if float(d.get("score", 0)) >= min_score
        ]

        return docs, top_score, candidates

    def execute_railway_tool(
        self,
        tool_name: str,
        arguments: Dict[str, Any]
    ) -> Dict[str, Any]:
        """
        Execute only approved public passenger tools.

        The LLM cannot directly specify arbitrary API URLs.
        """

        logger.info(
            f"🔧 Railway tool requested: {tool_name}"
        )

        try:

            # ====================================================
            # SEARCH STATIONS
            # ====================================================

            if tool_name == "search_stations":

                query = arguments.get("query", "")

                if not query or not str(query).strip():

                    return {
                        "success": False,
                        "error": (
                            "A station name or station code "
                            "is required."
                        )
                    }

                limit = arguments.get("limit", 10)

                return self.railway_api.search_stations(
                    query=str(query),
                    limit=int(limit)
                )

            # ====================================================
            # GET STATION
            # ====================================================

            elif tool_name == "get_station":

                station_id = arguments.get("station_id")

                if station_id is None:

                    return {
                        "success": False,
                        "error": "station_id is required."
                    }

                return self.railway_api.get_station(
                    int(station_id)
                )

            # ====================================================
            # LIST TRAINS
            # ====================================================

            elif tool_name == "list_trains":

                status = arguments.get("status", "ACTIVE")
                limit = arguments.get("limit", 50)

                return self.railway_api.list_trains(
                    status=status,
                    limit=int(limit)
                )

            # ====================================================
            # SEARCH TRAINS
            # ====================================================

            elif tool_name == "search_trains":

                query = arguments.get("query", "")

                if not query or not str(query).strip():
                    return {
                        "success": False,
                        "error": "A train number or train name is required."
                    }

                status = arguments.get("status", "ACTIVE")
                limit = arguments.get("limit", 10)

                return self.railway_api.search_trains(
                    query=str(query).strip(),
                    status=status,
                    limit=int(limit)
                )

            # ====================================================
            # GET TRAIN
            # ====================================================

            elif tool_name == "get_train":

                train_id = arguments.get("train_id")

                if train_id is None:

                    return {
                        "success": False,
                        "error": "train_id is required."
                    }

                return self.railway_api.get_train(
                    int(train_id)
                )

            # ====================================================
            # GET TRAIN STOPS
            # ====================================================

            elif tool_name == "get_train_stops":

                train_id = arguments.get("train_id")

                if train_id is None:

                    return {
                        "success": False,
                        "error": "train_id is required."
                    }

                return self.railway_api.get_train_stops(
                    int(train_id)
                )

            # ====================================================
            # GET ROUTE
            # ====================================================

            elif tool_name == "get_route":

                route_id = arguments.get("route_id")

                if route_id is None:

                    return {
                        "success": False,
                        "error": "route_id is required."
                    }

                return self.railway_api.get_route(
                    int(route_id)
                )

            # ====================================================
            # GET TRAINS BY ROUTE
            # ====================================================

            elif tool_name == "get_trains_by_route":

                route_id = arguments.get("route_id")

                if route_id is None:

                    return {
                        "success": False,
                        "error": "route_id is required."
                    }

                status = arguments.get("status")

                return self.railway_api.get_trains_by_route(
                    route_id=int(route_id),
                    status=status
                )

            # ====================================================
            # SEARCH DYNAMIC SCHEDULES
            # ====================================================

            elif tool_name == "search_schedules":

                from_station_id = arguments.get("from_station_id")
                to_station_id = arguments.get("to_station_id")
                route_ids = arguments.get("route_ids")
                date_from = arguments.get("date_from")
                date_to = arguments.get("date_to")

                return self.railway_api.search_schedules(
                    from_station_id=int(from_station_id),
                    to_station_id=int(to_station_id),
                    route_ids=route_ids,
                    date_from=date_from,
                    date_to=date_to,
                )

            # ====================================================
            # CALCULATE EXACT CONFIGURED FARE
            # ====================================================

            elif tool_name == "calculate_fare":

                train_id = arguments.get("train_id")
                from_rs_id = arguments.get("from_route_station_id")
                to_rs_id = arguments.get("to_route_station_id")

                if (
                    train_id is None
                    or from_rs_id is None
                    or to_rs_id is None
                ):
                    return {
                        "success": False,
                        "error": (
                            "train_id, from_route_station_id and "
                            "to_route_station_id are required."
                        )
                    }

                return self.railway_api.calculate_fare(
                    train_id=int(train_id),
                    from_route_station_id=int(from_rs_id),
                    to_route_station_id=int(to_rs_id),
                    class_type=arguments.get(
                        "class_type",
                        "ECONOMY_CLASS"
                    ),
                    seat_type=arguments.get("seat_type"),
                )

            # ====================================================
            # GET FARE PRICE MATRIX
            # ====================================================

            elif tool_name == "get_fare_price_matrix":

                train_id = arguments.get("train_id")

                if train_id is None:
                    return {
                        "success": False,
                        "error": "train_id is required."
                    }

                return self.railway_api.get_fare_price_matrix(
                    train_id=int(train_id),
                    class_type=arguments.get(
                        "class_type",
                        "ECONOMY_CLASS"
                    ),
                )

            # ====================================================
            # CALCULATE YANGON-PYAY SMALL-CONSIGNMENT PARCEL FEE
            # ====================================================

            elif tool_name == "calculate_parcel_fee":
                return self.railway_api.calculate_parcel_fee(
                    route_id=int(arguments.get("route_id")),
                    from_station_id=int(arguments.get("from_station_id")),
                    to_station_id=int(arguments.get("to_station_id")),
                    weight_viss=float(arguments.get("weight_viss")),
                )

            # ====================================================
            # GET SCHEDULE SEAT MAP
            # ====================================================

            elif tool_name == "get_schedule_seat_map":

                schedule_id = arguments.get("schedule_id")

                if schedule_id is None:
                    return {
                        "success": False,
                        "error": "schedule_id is required."
                    }

                return self.railway_api.get_schedule_seat_map(
                    schedule_id=int(schedule_id)
                )

            # ====================================================
            # CHECK ONE SEAT ON ONE SCHEDULE
            # ====================================================

            elif tool_name == "check_seat_availability":

                seat_id = arguments.get("seat_id")
                schedule_id = arguments.get("schedule_id")

                if seat_id is None or schedule_id is None:
                    return {
                        "success": False,
                        "error": "seat_id and schedule_id are required."
                    }

                return self.railway_api.check_seat_availability(
                    seat_id=int(seat_id),
                    schedule_id=int(schedule_id),
                )

            # ====================================================
            # READ-ONLY TICKET / BOOKING JOURNEY STATUS
            # ====================================================

            elif tool_name == "get_ticket_journey_status":

                ticket_no = arguments.get("ticket_no")

                if not ticket_no or not str(ticket_no).strip():
                    return {
                        "success": False,
                        "error": "A ticket number is required."
                    }

                return self.railway_api.get_ticket_journey_status(
                    ticket_no=str(ticket_no).strip()
                )

            # ====================================================
            # GET ROUTE TRAIN SCHEDULE
            # ====================================================

            elif tool_name == "get_route_train_schedule":

                route_id = arguments.get("route_id")
                train_id = arguments.get("train_id")

                if route_id is None or train_id is None:

                    return {
                        "success": False,
                        "error": (
                            "route_id and train_id "
                            "are required."
                        )
                    }

                return self.railway_api.get_route_train_schedule(
                    route_id=int(route_id),
                    train_id=int(train_id)
                )

            # ====================================================
            # GET NEXT TRAIN
            # ====================================================

            elif tool_name == "get_next_train":

                route_id = arguments.get("route_id")
                train_id = arguments.get("train_id")
                station_order = arguments.get("station_order")

                if route_id is None or train_id is None or station_order is None:

                    return {
                        "success": False,
                        "error": (
                            "route_id, train_id and station_order "
                            "are required."
                        )
                    }

                return self.railway_api.get_next_train(
                    route_id=int(route_id),
                    train_id=int(train_id),
                    station_order=int(station_order)
                )

            # ====================================================
            # UNKNOWN TOOL
            # ====================================================

            else:

                logger.warning(
                    f"🚫 Unauthorized railway tool: {tool_name}"
                )

                return {
                    "success": False,
                    "error": "This railway operation is not available."
                }

        except Exception as e:

            logger.error(
                f"❌ Tool execution failed: {e}",
                exc_info=True
            )

            return {
                "success": False,
                "error": "Railway information could not be retrieved.",
                "details": str(e)
            }


    @staticmethod
    def _looks_like_pseudo_tool_call(text: str) -> bool:
        """Detect a printed function call that should have been native."""
        if not text:
            return False

        lowered = text.casefold()
        compact = lowered.replace(" ", "")

        tool_names = [
            "search_schedules",
            "search_stations",
            "search_trains",
            "get_station",
            "get_train",
            "get_train_stops",
            "get_route",
            "get_trains_by_route",
            "get_route_train_schedule",
            "get_next_train",
            "calculate_fare",
            "get_fare_price_matrix",
            "calculate_parcel_fee",
            "calculate_parcel_fee",
            "get_schedule_seat_map",
            "check_seat_availability",
            "get_ticket_journey_status",
        ]

        return any(
            (
                f"{name}(" in lowered
                or f"{name} (" in lowered
                or f'"name":"{name}"' in compact
                or f"'name':'{name}'" in compact
            )
            for name in tool_names
        )

    @staticmethod
    def _contains_internal_output_leak(text: str) -> bool:
        """Detect internal implementation details in passenger-facing text."""
        if not text:
            return False

        lowered = text.casefold()

        patterns = [
            r"\bstation[_\s-]*id\b",
            r"\btrain[_\s-]*id\b",
            r"\broute[_\s-]*id\b",
            r"\bschedule[_\s-]*id\b",
            r"\broute[_\s-]*station[_\s-]*id\b",
            r"\border\s*\d+\b",
            r"\bsearch_schedules\s*\(",
            r"\bsearch_stations\s*\(",
            r"\bsearch_trains\s*\(",
            r"\bget_train_stops\s*\(",
            r"\bget_route_train_schedule\s*\(",
            r"\bget_next_train\s*\(",
            r"\btool[_\s-]*call\b",
        ]

        return any(re.search(pattern, lowered) for pattern in patterns)

    def _rewrite_passenger_answer(
        self,
        messages: list,
        provider: str,
    ) -> str:
        rewrite_messages = list(messages)
        rewrite_messages.append({
            "role": "user",
            "content": (
                "Using ONLY the railway tool results already present above, "
                "return the final passenger-facing answer. Do not mention internal "
                "IDs, route order, JSON, tool/API names, function calls, parameters "
                "or hidden reasoning. If there is no matching service, simply say "
                "that no matching train service was found."
            )
        })

        result = self.llm.generate_turn(
            messages=rewrite_messages,
            tools=RAILWAY_TOOLS,
            provider=provider,
            tool_choice="none",
            max_new_tokens=Config.LLM_MAX_TOKENS,
        )

        return result["text"]

    def run_tool_loop(
        self,
        prompt: str,
        model_mode: str = "Auto",
        max_tool_calls: int = 4
    ):
        """
        Provider-neutral native tool loop.

        Auto:
          Gemini is locked for the turn if available.
          If a Gemini API request fails, the same normalized conversation
          switches to Qwen and stays on Qwen for the rest of that turn.

        Gemini / Qwen:
          locked to the explicitly selected provider.
        """
        messages = [{
            "role": "user",
            "content": prompt,
        }]

        provider = self.llm.choose_provider(
            model_mode
        )
        requested_mode = (
            HybridLLMService.normalize_model_mode(
                model_mode
            )
        )

        tool_used = False
        executed_calls = 0
        next_tool_choice = "auto"
        pseudo_retry_used = False
        fallback_used = False

        for iteration in range(max_tool_calls + 2):
            print(
                f"\n🔧 Native tool iteration: {iteration + 1} | "
                f"provider={provider} | mode={requested_mode}"
            )

            try:
                turn = self.llm.generate_turn(
                    messages=messages,
                    tools=RAILWAY_TOOLS,
                    provider=provider,
                    tool_choice=next_tool_choice,
                    max_new_tokens=Config.LLM_MAX_TOKENS,
                )

            except Exception as provider_error:
                fallback = self.llm.fallback_provider(
                    provider,
                    requested_mode
                )

                if fallback and not fallback_used:
                    print(
                        "⚠️ Gemini request failed in Auto mode; "
                        "switching this turn to Qwen fallback."
                    )
                    provider = fallback
                    fallback_used = True
                    turn = self.llm.generate_turn(
                        messages=messages,
                        tools=RAILWAY_TOOLS,
                        provider=provider,
                        tool_choice=next_tool_choice,
                        max_new_tokens=Config.LLM_MAX_TOKENS,
                    )
                else:
                    raise provider_error

            next_tool_choice = "auto"

            text = turn.get("text", "")
            tool_calls = list(
                turn.get("tool_calls") or []
            )

            # Model printed a pseudo tool call instead of making it natively.
            if not tool_calls and self._looks_like_pseudo_tool_call(text):
                if not pseudo_retry_used:
                    print(
                        "⚠️ Pseudo tool call detected; forcing a native call."
                    )
                    messages.append({
                        "role": "user",
                        "content": (
                            "Do not print or describe a function call. "
                            "Make the appropriate native railway tool call now. "
                            "If a required passenger detail is missing, ask one "
                            "short clarification instead."
                        )
                    })
                    next_tool_choice = "required"
                    pseudo_retry_used = True
                    continue

            if not tool_calls:
                final_text = text

                # Hard passenger-output gate.
                if self._contains_internal_output_leak(
                    final_text
                ):
                    print(
                        "⚠️ Internal implementation detail detected; "
                        "rewriting passenger answer."
                    )
                    final_text = self._rewrite_passenger_answer(
                        messages,
                        provider,
                    )

                return (
                    final_text,
                    tool_used,
                    provider,
                    fallback_used,
                )

            # Preserve the assistant native-call turn.
            assistant_turn = {
                "role": "assistant",
                "content": text,
                "tool_calls": tool_calls,
            }

            # For Gemini, retain the exact SDK Content returned by the model.
            # This carries Gemini 3/3.5 thought_signature metadata.
            if (
                turn.get("provider") == "gemini"
                and turn.get("gemini_content") is not None
            ):
                assistant_turn["gemini_content"] = (
                    turn["gemini_content"]
                )
                print(
                    "🧠 Preserved Gemini function-call Content "
                    "(thought signature retained)"
                )

            messages.append(assistant_turn)

            for call in tool_calls:
                if executed_calls >= max_tool_calls:
                    break

                candidate = {
                    "name": call["name"],
                    "arguments": dict(
                        call.get("arguments") or {}
                    ),
                }

                valid, error = validate_tool_call(
                    candidate
                )

                if valid:
                    print(
                        f"\n🔧 Executing native tool: "
                        f"{candidate['name']}"
                    )
                    print(
                        "📦 Arguments:",
                        candidate["arguments"]
                    )

                    result = self.execute_railway_tool(
                        candidate["name"],
                        candidate["arguments"],
                    )
                else:
                    result = {
                        "success": False,
                        "error": error,
                        "instruction": (
                            "Do not invent missing identifiers. "
                            "Ask one short passenger clarification "
                            "when genuinely needed."
                        ),
                    }

                llm_result = compact_tool_result_for_llm(
                    candidate["name"],
                    result
                )

                print("\n📡 Railway API result (LLM-safe/compact):")
                print(
                    json.dumps(
                        llm_result,
                        ensure_ascii=False,
                        indent=2,
                        default=str,
                    )
                )

                messages.append({
                    "role": "tool",
                    "tool_call_id": call["id"],
                    "name": candidate["name"],
                    "content": json.dumps(
                        llm_result,
                        ensure_ascii=False,
                        default=str,
                    ),
                })

                tool_used = True
                executed_calls += 1

            if executed_calls >= max_tool_calls:
                final_text = self._rewrite_passenger_answer(
                    messages,
                    provider,
                )
                return (
                    final_text,
                    tool_used,
                    provider,
                    fallback_used,
                )

        final_text = self._rewrite_passenger_answer(
            messages,
            provider,
        )
        return (
            final_text,
            tool_used,
            provider,
            fallback_used,
        )

    def chat(

        self,
        question_mm: str,
        history: Optional[List[Dict]] = None,
        min_score: Optional[float] = None,
        model_mode: str = "Auto"
    ) -> Dict:
        """
        Process one passenger message.

        Despite the legacy parameter name `question_mm`, English and
        Myanmar are both accepted.
        """
        start_time = time.time()
        history = history or []
        model_mode = HybridLLMService.normalize_model_mode(
            model_mode
        )
        min_score = (
            Config.MIN_SCORE
            if min_score is None
            else min_score
        )

        question = (question_mm or "").strip()

        if not question:
            return {
                "response": "မေးခွန်းတစ်ခု ရိုက်ထည့်ပေးပါ။",
                "mode": "validation",
                "intent": None,
                "confidence": 0.0,
                "sources": [],
                "retrieved_chunks": 0,
                "from_cache": False,
                "response_time_ms": 0,
                "translation_used": False,
                "translation_model": None,
                "translation_time_ms": 0,
                "llm_model": None,
                "llm_provider": None,
                "requested_model_mode": model_mode,
                "tool_used": False,
            }

        try:
            # ========================================================
            # STEP 0: CACHE
            # ========================================================
            cache_key = self._cache_key(
                question,
                history,
                model_mode
            )

            if (
                Config.ENABLE_CACHE
                and cache_key in self.response_cache
            ):
                cached = self.response_cache[
                    cache_key
                ].copy()

                cached["from_cache"] = True

                print(
                    "✅ Response from cache: "
                    f"{cached['response'][:60]}..."
                )

                return cached

            # ========================================================
            # STEP 1: DIRECT MULTILINGUAL RETRIEVAL
            # ========================================================
            print("🔍 Step 1: BGE-M3 direct multilingual retrieval...")
            print(f"📝 Original question: {question}")

            direct_docs, direct_top_score, _ = self._retrieve(
                question,
                min_score
            )

            docs = direct_docs
            selected_top_score = direct_top_score
            retrieval_query = question

            translation_used = False
            translation_time_ms = 0
            translated_query = None

            print(
                "   Direct top score: "
                f"{direct_top_score:.4f}"
            )

            # ========================================================
            # STEP 2: OPTIONAL ENGLISH RETRIEVAL FALLBACK
            # ========================================================
            should_try_translation = (
                Config.TRANSLATION_FALLBACK_ENABLED
                and contains_myanmar(question)
                and direct_top_score
                    < Config.TRANSLATION_FALLBACK_SCORE
            )

            if should_try_translation:
                print(
                    "🌐 Direct retrieval is weak; trying "
                    "Myanmar -> English retrieval fallback..."
                )

                translate_start = time.time()

                translated_query = self.translator.mm_to_en(
                    question
                )

                translation_time_ms = int(
                    (time.time() - translate_start) * 1000
                )

                if (
                    translated_query
                    and translated_query.strip()
                    and translated_query.strip() != question
                ):
                    (
                        fallback_docs,
                        fallback_top_score,
                        _
                    ) = self._retrieve(
                        translated_query,
                        min_score
                    )

                    print(
                        "   Fallback top score: "
                        f"{fallback_top_score:.4f}"
                    )

                    # Use translated retrieval only when it actually
                    # improves the retrieval result.
                    if (
                        fallback_docs
                        and (
                            not docs
                            or fallback_top_score
                                > selected_top_score + 0.01
                        )
                    ):
                        docs = fallback_docs
                        selected_top_score = fallback_top_score
                        retrieval_query = translated_query
                        translation_used = True

                        print(
                            "✅ English retrieval fallback selected."
                        )
                    else:
                        print(
                            "ℹ️ Direct multilingual retrieval kept."
                        )

            # ========================================================
            # RETRIEVAL LOG
            # ========================================================
            print("\n" + "=" * 60)
            print(f"📚 Retrieved Documents: {len(docs)}")
            print(
                "🌐 Translation fallback used: "
                f"{translation_used}"
            )

            for i, doc in enumerate(docs[:3], 1):
                print(
                    f"  {i}. "
                    f"[{doc.get('category', 'N/A')}] "
                    f"{doc.get('title', 'Untitled')} "
                    f"(Score: {float(doc.get('score', 0)):.4f})"
                )

            print("=" * 60 + "\n")

            # ========================================================
            # STEP 3: BUILD PROMPT WITH ORIGINAL QUESTION
            # ========================================================
            print("🧩 Step 3: Building multilingual prompt...")

            prompt = self.prompt_builder.build_prompt(
                question=question,
                documents=docs,
                history=history,
                tool_results=""
            )

            # ========================================================
            # STEP 4: SELECTED MODEL + OPTIONAL RAILWAY TOOL LOOP
            # ========================================================
            print(
                "🤖 Step 4: Generating with model mode:",
                model_mode
            )

            llm_start = time.time()
            tool_used = False
            provider_used = None
            provider_fallback_used = False

            try:
                (
                    response_text,
                    tool_used,
                    provider_used,
                    provider_fallback_used,
                ) = self.run_tool_loop(
                    prompt=prompt,
                    model_mode=model_mode,
                    max_tool_calls=4
                )

            except Exception as tool_error:
                logger.exception(
                    "Tool-enabled generation failed: %s",
                    tool_error
                )

                # Dynamic railway facts must never be guessed by a tool-free
                # fallback. If native tool calling itself fails, fail closed.
                if requires_dynamic_railway_api(question):
                    response_text = operational_data_unavailable_message(
                        question
                    )
                    tool_used = False
                else:
                    response_text = self.llm.generate_direct(
                        prompt,
                        model_mode=model_mode,
                        max_new_tokens=Config.LLM_MAX_TOKENS
                    )
                    provider_used = self.llm.last_provider_used

            # --------------------------------------------------------
            # OPERATIONAL FACT SAFETY GATE
            # --------------------------------------------------------
            # Even if the model returns a fluent answer without raising an
            # exception, an operational question is not allowed to escape
            # without a successful backend tool call.
            #
            # Operational values must never escape from RAG/model memory
            # when the backend path was required.
            if (
                requires_dynamic_railway_api(question)
                and not tool_used
                and not is_safe_operational_clarification(
                    response_text
                )
            ):
                logger.warning(
                    "Blocked tool-free operational railway answer."
                )
                response_text = operational_data_unavailable_message(
                    question
                )

            llm_time_ms = int(
                (time.time() - llm_start) * 1000
            )

            # ========================================================
            # STEP 5: NO FINAL TRANSLATION
            # ========================================================
            if not response_text:
                if contains_myanmar(question):
                    response_text = (
                        "တောင်းပန်ပါတယ်။ လက်ရှိအချိန်မှာ "
                        "အချက်အလက်ရယူပြီး ဖြေကြားပေးနိုင်ခြင်း "
                        "မရှိသေးပါဘူး။"
                    )
                else:
                    response_text = (
                        "Sorry, I couldn't retrieve enough information "
                        "to answer that right now."
                    )

            # ========================================================
            # STEP 6: SOURCES + METADATA
            # ========================================================
            sources = [
                {
                    "title": d.get("title", "Unknown"),
                    "dataset": d.get("dataset"),
                    "category": d.get(
                        "category",
                        "general"
                    ),
                    "language": d.get("language"),
                    "source_type": d.get("source_type"),
                    "source": d.get("source"),
                    "source_url": d.get("source_url"),
                    "as_of_date": d.get("as_of_date"),
                    "content_preview": d.get(
                        "content",
                        ""
                    )[:200],
                    "score": round(
                        float(d.get("score", 0)),
                        4
                    ),
                    "relevance": (
                        "high"
                        if float(d.get("score", 0)) > 0.75
                        else "medium"
                    ),
                }
                for d in docs
            ]

            if tool_used and docs:
                mode = "rag_tool"
            elif tool_used:
                mode = "tool"
            elif docs:
                mode = "rag"
            else:
                mode = "llm"

            confidence = (
                round(
                    float(docs[0].get("score", 0)),
                    4
                )
                if docs
                else 0.0
            )

            result = {
                "response": response_text.strip(),
                "mode": mode,
                "intent": None,
                "confidence": confidence,
                "sources": sources,
                "retrieved_chunks": len(docs),
                "from_cache": False,
                "response_time_ms": int(
                    (time.time() - start_time) * 1000
                ),

                # Translation now means retrieval fallback only.
                "translation_used": translation_used,
                "translation_model": (
                    "deep-translator (retrieval fallback only)"
                    if translation_used
                    else None
                ),
                "translation_time_ms": translation_time_ms,

                # Diagnostic fields
                "retrieval_query": retrieval_query,
                "translated_query": translated_query,
                "direct_retrieval_score": round(
                    direct_top_score,
                    4
                ),
                "selected_retrieval_score": round(
                    selected_top_score,
                    4
                ),
                "response_language": (
                    "my"
                    if contains_myanmar(question)
                    else "en"
                ),
                "llm_model": self.llm.last_model_used,
                "llm_provider": (
                    provider_used
                    or self.llm.last_provider_used
                ),
                "requested_model_mode": model_mode,
                "provider_fallback_used": provider_fallback_used,
                "llm_time_ms": llm_time_ms,
                "tool_used": tool_used,
            }

            # Cache only stable/non-operational answers.
            # Never cache schedules, train-stop answers, fares, seats, or booking status.
            if (
                Config.ENABLE_CACHE
                and not tool_used
                and not requires_dynamic_railway_api(question)
            ):
                self.response_cache[
                    cache_key
                ] = result

            print(
                "\n📊 LLM Stats:",
                self.llm.get_stats()
            )
            print(
                "📊 Translation fallback used:",
                translation_used
            )
            print(
                "📊 Mode:",
                mode
            )

            return result

        except Exception as e:
            logger.exception(
                "Error in chat: %s",
                e
            )

            error_response = (
                "တောင်းပန်ပါတယ်။ စနစ်တွင် ချို့ယွင်းချက် "
                "ရှိနေပါသည်။ နောက်မှ ထပ်မံကြိုးစားပေးပါ။"
                if contains_myanmar(question)
                else
                "Sorry, the system encountered an error. "
                "Please try again."
            )

            return {
                "response": error_response,
                "mode": "error",
                "intent": None,
                "confidence": 0.0,
                "sources": [],
                "retrieved_chunks": 0,
                "from_cache": False,
                "response_time_ms": int(
                    (time.time() - start_time) * 1000
                ),
                "translation_used": False,
                "translation_model": None,
                "translation_time_ms": 0,
                "llm_model": (
                    self.llm.last_model_used
                    if self._llm is not None
                    else None
                ),
                "llm_provider": (
                    self.llm.last_provider_used
                    if self._llm is not None
                    else None
                ),
                "requested_model_mode": model_mode,
                "tool_used": False,
                "error": str(e),
            }

    def clear_cache(self):
        self.response_cache.clear()

        if self._translator is not None:
            self._translator.clear_cache()

        print("🧹 All caches cleared")

    def get_stats(self) -> dict:
        stats = {
            "cache_size": len(self.response_cache),
            "memory": self.memory_monitor.get_summary(),
            "translation_service_loaded": (
                self._translator is not None
            ),
        }

        if self._translator is not None:
            stats["translation"] = (
                self._translator.get_stats()
            )

        if self._llm is not None:
            stats["llm"] = self._llm.get_stats()

        return stats

    def __del__(self):
        for attr in [
            "_embedding",
            "_mongo",
            "_translator",
            "_llm"
        ]:
            if hasattr(self, attr):
                obj = getattr(self, attr)

                if (
                    obj is not None
                    and hasattr(obj, "__del__")
                ):
                    try:
                        obj.__del__()
                    except Exception:
                        pass

                setattr(self, attr, None)

        if hasattr(self, "response_cache"):
            self.response_cache.clear()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        gc.collect()


In [ ]:
# ============================================================
# REFRESH STABLE KNOWLEDGE AFTER NEON REFERENCE-DATA CHANGES
# ============================================================

def refresh_railway_catalog(path: str = None):
    """
    Run only after stable train/station/route records change.

    1) Re-export railway_catalog.json from Neon.
    2) Replace /content/railway_catalog.json.
    3) Run refresh_railway_catalog() once.

    Passenger questions do NOT read the file.
    """
    global railway_catalog
    global RAILWAY_SYSTEM_KNOWLEDGE

    if path:
        railway_catalog.path = Path(path)

    railway_catalog.reload()

    RAILWAY_SYSTEM_KNOWLEDGE = (
        railway_catalog.system_knowledge
        if railway_catalog.loaded
        else ""
    )

    engine = getattr(
        RAGEngine,
        "_instance",
        None
    )

    if engine is not None:
        engine._railway_catalog = railway_catalog

        if engine._llm is not None:
            engine._llm.update_stable_knowledge(
                RAILWAY_SYSTEM_KNOWLEDGE
            )

        engine.response_cache.clear()
        print(
            "🧹 Response cache cleared after "
            "stable-knowledge refresh"
        )

    return railway_catalog


print(
    "✅ Stable-knowledge refresh helper ready: "
    "refresh_railway_catalog()"
)


✅ Stable-knowledge refresh helper ready: refresh_railway_catalog()


### Updating stable railway records later

The JSON is now a **startup/reference source**, not per-question context. When trains, stations, or routes change: re-export from Neon, replace `/content/railway_catalog.json`, then run `refresh_railway_catalog()` once. The in-memory system knowledge is rebuilt for both Gemini and Qwen. Schedules remain live FastAPI data.

In [ ]:
# ============================================================
# v11 STABLE KNOWLEDGE + OPERATIONAL TOOL CONTRACT SMOKE TEST
# ============================================================

print("Catalog loaded:", railway_catalog.loaded)
print(
    "Stable system knowledge chars:",
    len(railway_catalog.system_knowledge)
)

assert railway_catalog.loaded
assert "TRAINS:" in railway_catalog.system_knowledge
assert "ROUTES:" in railway_catalog.system_knowledge
assert "STATIONS:" in railway_catalog.system_knowledge

# Verify known demo references are represented internally.
assert "S9=မှော်ဘီ ဘူတာ" in railway_catalog.system_knowledge
assert "S7=ပြည် ဘူတာ" in railway_catalog.system_knowledge
assert "R5=" in railway_catalog.system_knowledge
assert "T6=" in railway_catalog.system_knowledge

contract_examples = [
    {
        "name": "search_schedules",
        "arguments": {
            "from_station_id": 9,
            "to_station_id": 7,
            "route_ids": [5, 16],
            "date_from": "2026-08-25",
            "date_to": "2026-08-25",
        },
    },
    {
        "name": "calculate_fare",
        "arguments": {
            "train_id": 6,
            # EXAMPLE RouteStation IDs only; this test validates schema,
            # it does not execute the backend request.
            "from_route_station_id": 1,
            "to_route_station_id": 2,
            "class_type": "ECONOMY_CLASS",
        },
    },
    {
        "name": "calculate_parcel_fee",
        "arguments": {
            "route_id": 5,
            "from_station_id": 9,
            "to_station_id": 7,
            "weight_viss": 10,
        },
    },
    {
        "name": "get_schedule_seat_map",
        "arguments": {"schedule_id": 1},
    },
    {
        "name": "get_ticket_journey_status",
        "arguments": {"ticket_no": "TKT-EXAMPLE"},
    },
]

for example_call in contract_examples:
    valid, error = validate_tool_call(example_call)
    print(
        f"{example_call['name']} schema valid:",
        valid,
        error,
    )
    assert valid

assert len(RAILWAY_TOOLS) == 17
print("✅ v12 stable knowledge and operational + parcel tool contracts passed.")


Catalog loaded: True
Stable system knowledge chars: 5725
search_schedules schema valid: True None
calculate_fare schema valid: True None
calculate_parcel_fee schema valid: True None
get_schedule_seat_map schema valid: True None
get_ticket_journey_status schema valid: True None
✅ v12 stable knowledge and operational + parcel tool contracts passed.


In [ ]:
# ============================================================
# v12 PARCEL CALCULATION RULE SMOKE TEST
# ============================================================

def _parcel_rule_test(actual_distance_miles: float, actual_weight_viss: float):
    chargeable_distance = max(float(actual_distance_miles), 100.0)
    chargeable_weight = max(float(actual_weight_viss), 5.0)
    raw = 0.13 * chargeable_distance * chargeable_weight
    final = int(math.ceil(raw / 50.0) * 50)
    return chargeable_distance, chargeable_weight, raw, final

# 2 viss, 40 miles -> charge as 5 viss x 100 miles:
assert _parcel_rule_test(40, 2) == (100.0, 5.0, 65.0, 100)

# 10 viss, 120 miles -> 156 MMK -> round to 200 MMK:
assert _parcel_rule_test(120, 10) == (120.0, 10.0, 156.0, 200)

print("✅ v12 parcel arithmetic tests passed")


✅ v12 parcel arithmetic tests passed


In [ ]:
# ============================================================
# v11 MODEL ROUTING / NATIVE TOOL SMOKE TESTS
# ============================================================
# Run after the backend and both provider credentials are available.
#
# RAGEngine.reset()
# rag_engine = RAGEngine.get_instance()
#
# Suggested tests:
# tests = [
#     ("Auto", "မှော်ဘီကနေ ပြည်ကို မနက်ဖြန် ဘယ်ရထားတွေရှိလဲ"),
#     ("Gemini", "Train 73 stops at which stations?"),
#     ("Qwen", "What is the ordinary fare from Yangon to Pyay on train 71?"),
#     ("Auto", "မနက်ဖြန် ရန်ကုန်ကနေ ပြည်သွားမယ့် ရထားမှာ ခုံလွတ်ရှိလား"),
#     ("Auto", "Track ticket TKT-EXAMPLE"),
#     ("Auto", "မှော်ဘီကနေ ပြည်ကို ပါဆယ် ၁၀ ပိဿာ ပို့ရင် ဘယ်လောက်ကျမလဲ"),
#     ("Qwen", "How much to send a 3-viss parcel from Yangon to Hmawbi?"),
# ]
#
# for model_mode, question in tests:
#     print("\\n" + "=" * 70)
#     print("MODEL MODE:", model_mode)
#     print("QUESTION:", question)
#     result = rag_engine.chat(
#         question_mm=question,
#         history=[],
#         model_mode=model_mode,
#     )
#     print("ANSWER:", result["response"])
#     print(
#         "PROVIDER:",
#         result.get("llm_provider"),
#         "| MODEL:",
#         result.get("llm_model"),
#         "| TOOL:",
#         result.get("tool_used"),
#         "| FALLBACK:",
#         result.get("provider_fallback_used"),
#     )


In [ ]:
# ============================================
# USAGE EXAMPLE
# ============================================

if __name__ == "__main__":
    print("\n🚀 Creating RAG Engine instance...")
    rag_engine = RAGEngine.get_instance()

    print("\n🧪 Testing direct Myanmar question...")
    result = rag_engine.chat(
        question_mm="ရန်ကုန်မှ ပဲခူးသို့ ရထားရှိလား",
        history=[]
    )

    print("\n" + "=" * 60)
    print("RESPONSE:")
    print("=" * 60)
    print(result["response"])
    print("\n" + "=" * 60)
    print(f"Mode: {result['mode']}")
    print(f"Confidence: {result['confidence']}")
    print(f"Response Time: {result['response_time_ms']}ms")
    print(f"LLM Time: {result.get('llm_time_ms', 0)}ms")
    print(
        "Translation fallback used:",
        result.get("translation_used", False)
    )
    print(
        "Translation fallback time:",
        result.get("translation_time_ms", 0),
        "ms"
    )
    print(f"LLM Model: {result.get('llm_model')}")
    print(f"Sources: {result.get('retrieved_chunks', 0)}")
    print("=" * 60)

    print("\n📊 Final Stats:")
    print(rag_engine.get_stats())



🚀 Creating RAG Engine instance...
Initializing Railway AI...
✅ RAG Engine initialized
✅ LLM routing: Gemini primary -> Qwen fallback (Auto mode)
✅ Direct multilingual RAG: enabled
✅ Translation: retrieval fallback only

🧪 Testing direct Myanmar question...
🔍 Step 1: BGE-M3 direct multilingual retrieval...
📝 Original question: ရန်ကုန်မှ ပဲခူးသို့ ရထားရှိလား
🔄 Loading BGE-M3 on CPU...
Loading BGE-M3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding model ready.
🔄 Loading MongoDB Service...
✅ MongoDB KB connected: railway_chatbot.documents_v4_1 (997 documents)
   Direct top score: 0.0000
🌐 Direct retrieval is weak; trying Myanmar -> English retrieval fallback...
🔄 Loading retrieval translation fallback...
Loading Translation Fallback Service...
✅ Translation fallback ready
   Used only for weak Myanmar RAG retrieval


ERROR:__main__:Translation fallback error (my->en): ရန်ကုန်မှ ပဲခူးသို့ ရထားရှိလား --> No translation was found using the current translator. Try another translator?



📚 Retrieved Documents: 0
🌐 Translation fallback used: False

🧩 Step 3: Building multilingual prompt...
📦 Dynamic prompt size: 675 chars (stable railway knowledge is in system context)
🤖 Step 4: Generating with model mode: Auto
🔄 Initializing Gemini/Qwen hybrid LLM service...
Initializing Gemini-primary hybrid LLM service...
✅ Gemini: gemini-3.5-flash via Vertex AI
✅ Qwen fallback: qwen/qwen3.6-27b via Groq
✅ Default: Auto

🔧 Native tool iteration: 1 | provider=gemini | mode=Auto


🧠 Preserved Gemini function-call Content (thought signature retained)

🔧 Executing native tool: search_stations
📦 Arguments: {'query': 'ပဲခူး'}

📡 Railway API result (LLM-safe/compact):
{
  "success": true,
  "data": [
    {
      "name": "ပဲခူး ဘူတာ",
      "code": "BG-MAIN074",
      "city": "ပဲခူးမြို့",
      "state_region": "ပဲခူး တိုင်းဒေသကြီး",
      "latitude": 17.334932,
      "longitude": 96.474904,
      "is_active": true,
      "id": 11
    }
  ]
}

🔧 Native tool iteration: 2 | provider=gemini | mode=Auto

📊 LLM Stats: {'total_calls': 2, 'api_attempts': 2, 'avg_time_ms': 1463, 'total_time_ms': 2927, 'last_provider_used': 'gemini', 'last_model_used': 'gemini-3.5-flash', 'last_error': None, 'last_usage': {'cache_tokens_details': None, 'cached_content_token_count': None, 'candidates_token_count': 180, 'candidates_tokens_details': [{'modality': <MediaModality.TEXT: 'TEXT'>, 'token_count': 180}], 'prompt_token_count': 6942, 'prompt_tokens_details': [{'modality': <MediaModality.T

In [ ]:
import gradio as gr

In [ ]:
print(gr.__version__)

6.26.0


In [ ]:
import gradio as ip
ip.close_all()

In [ ]:
# ============================================================
# v9 GEMINI FUNCTION-CALLING DIAGNOSTIC
# ============================================================
# If Gemini is connected, startup should report gemini_available=True.
#
# During a Gemini native tool call you should now see:
#   🧠 Preserved Gemini function-call Content (thought signature retained)
#
# A railway API 400/404 happens AFTER Gemini was successfully reached and
# should be debugged as a railway parameter/data issue, not Gemini access.

print("✅ v9 Gemini thought-signature preservation patch loaded.")
print(
    "Qwen fallback prompt budget:",
    {
        "docs": Config.PROMPT_DOC_LIMIT,
        "doc_chars": Config.PROMPT_DOC_CHARS,
        "history_messages": Config.PROMPT_HISTORY_MESSAGES,
        "history_chars_each": Config.PROMPT_HISTORY_CHARS_PER_MESSAGE,
    }
)


✅ v9 Gemini thought-signature preservation patch loaded.
Qwen fallback prompt budget: {'docs': 3, 'doc_chars': 700, 'history_messages': 3, 'history_chars_each': 350}


In [ ]:
# ============================================================
# v15 FARE / KB CONTRACT SMOKE TEST
# ============================================================

_test_call = {
    "name": "calculate_fare",
    "arguments": {
        "train_id": 10,
        "from_route_station_id": 564,
        "to_route_station_id": 588,
        "class_type": "ORDINARY",
    },
}

_valid, _error = validate_tool_call(_test_call)

assert _valid, _error
assert _test_call["arguments"]["class_type"] == "ECONOMY_CLASS"

assert (
    RailwayAPIClient._normalize_fare_class("Economy Class")
    == "ECONOMY_CLASS"
)
assert (
    RailwayAPIClient._normalize_fare_class("ordinary")
    == "ECONOMY_CLASS"
)
assert (
    RailwayAPIClient._normalize_fare_class("upper")
    == "UPPER_CLASS"
)
assert (
    RailwayAPIClient._normalize_fare_class("sleeper")
    == "SLEEPER"
)

_fake_seat_map = {
    "success": True,
    "data": {
        "schedule_id": 123,
        "coaches": [
            {
                "id": 1,
                "name": "BC-001",
                "coach_type": "ECONOMY_CLASS",
                "order_number": 1,
                "seats": [
                    {"seat_number": "A1", "available": True},
                    {"seat_number": "A2", "available": False},
                ],
            }
        ],
    },
}

_compact = compact_tool_result_for_llm(
    "get_schedule_seat_map",
    _fake_seat_map
)

assert _compact["data"]["total_seats"] == 2
assert _compact["data"]["available_seats"] == 1
assert "seats" not in _compact["data"]["coaches"][0]

print("✅ v15 fare/API/KB smoke tests passed")
print("Configured knowledge collection:", Config.COLLECTION_NAME)


✅ v15 fare/API/KB smoke tests passed
Configured knowledge collection: documents_v4_1


In [ ]:
import logging
import time
from typing import Dict, Any, List
import gradio as gr

logger = logging.getLogger(__name__)

MODEL_CHOICES = [
    "Auto (Gemini → Qwen)",
    "Gemini",
    "Qwen",
]


def _normalize_ui_model_choice(choice: str) -> str:
    return HybridLLMService.normalize_model_mode(
        choice
    )


# ============================================
# CHAT HANDLER
# ============================================

def respond(
    message: str,
    chat_history: List[Dict[str, Any]],
    model_choice: str = "Auto (Gemini → Qwen)"
):
    if not message or not message.strip():
        return (
            "",
            chat_history or [],
            "⚠️ Please enter a message"
        )

    history = list(chat_history or [])
    model_mode = _normalize_ui_model_choice(
        model_choice
    )

    try:
        result = rag_engine.chat(
            question_mm=message,
            history=history,
            min_score=Config.MIN_SCORE,
            model_mode=model_mode,
        )

        response_text = result.get(
            "response",
            "No response generated"
        )

        history.append({
            "role": "user",
            "content": message
        })
        history.append({
            "role": "assistant",
            "content": response_text
        })

        status = (
            f"✅ {result.get('response_time_ms', 0)}ms | "
            f"Mode: {result.get('mode', 'unknown')} | "
            f"Provider: {result.get('llm_provider', 'unknown')} | "
            f"Model: {result.get('llm_model', 'unknown')} | "
            f"Fallback: "
            f"{'yes' if result.get('provider_fallback_used') else 'no'}"
        )

        return "", history, status

    except Exception as exc:
        logger.exception("Gradio chat failed")
        return "", history, f"❌ Error: {exc}"


# ============================================
# BACKWARD-COMPATIBLE API USED BY FASTAPI
# ============================================
# Existing AIAgentService can keep sending only:
#   message, chat_history
# Backend API always uses Auto = Gemini -> Qwen.

def json_respond(
    message: str,
    chat_history: List[Dict[str, Any]]
) -> Dict[str, Any]:
    start_time = time.time()

    try:
        result = rag_engine.chat(
            question_mm=(message or "").strip(),
            history=chat_history or [],
            min_score=Config.MIN_SCORE,
            model_mode="Auto",
        )

        return {
            "success": True,
            "response": result.get("response", ""),
            "mode": result.get("mode", "rag"),
            "intent": result.get("intent"),
            "confidence": result.get("confidence", 0.0),
            "sources": result.get("sources", []),
            "retrieved_chunks": result.get(
                "retrieved_chunks",
                0
            ),
            "translation_used": result.get(
                "translation_used",
                False
            ),
            "translation_model": result.get(
                "translation_model"
            ),
            "from_cache": result.get(
                "from_cache",
                False
            ),
            "response_time_ms": result.get(
                "response_time_ms",
                int(
                    (time.time() - start_time)
                    * 1000
                )
            ),
            "llm_model": result.get("llm_model"),
            "llm_provider": result.get(
                "llm_provider"
            ),
            "provider_fallback_used": result.get(
                "provider_fallback_used",
                False
            ),
        }

    except Exception as exc:
        logger.exception("/chat API failed")
        return {
            "success": False,
            "response": (
                "တောင်းပန်ပါတယ်။ AI တွင် "
                "အမှားတစ်ခု ဖြစ်ပွားခဲ့ပါသည်။"
            ),
            "mode": "error",
            "intent": None,
            "confidence": 0.0,
            "sources": [],
            "retrieved_chunks": 0,
            "translation_used": False,
            "translation_model": None,
            "from_cache": False,
            "response_time_ms": int(
                (time.time() - start_time) * 1000
            ),
            "llm_model": None,
            "llm_provider": None,
            "provider_fallback_used": False,
            "error": str(exc),
        }



# Optional API for a future FastAPI/React model selector.
# Existing /chat remains unchanged and uses Auto.
def json_respond_with_model(
    message: str,
    chat_history: List[Dict[str, Any]],
    model_choice: str = "Auto"
) -> Dict[str, Any]:
    start_time = time.time()
    model_mode = _normalize_ui_model_choice(
        model_choice
    )

    try:
        result = rag_engine.chat(
            question_mm=(message or "").strip(),
            history=chat_history or [],
            min_score=Config.MIN_SCORE,
            model_mode=model_mode,
        )

        return {
            "success": True,
            "response": result.get("response", ""),
            "mode": result.get("mode", "rag"),
            "confidence": result.get("confidence", 0.0),
            "sources": result.get("sources", []),
            "retrieved_chunks": result.get("retrieved_chunks", 0),
            "response_time_ms": result.get(
                "response_time_ms",
                int((time.time() - start_time) * 1000)
            ),
            "llm_model": result.get("llm_model"),
            "llm_provider": result.get("llm_provider"),
            "requested_model_mode": model_mode,
            "provider_fallback_used": result.get(
                "provider_fallback_used",
                False
            ),
        }

    except Exception as exc:
        return {
            "success": False,
            "response": "AI request failed.",
            "llm_model": None,
            "llm_provider": None,
            "requested_model_mode": model_mode,
            "provider_fallback_used": False,
            "error": str(exc),
        }


# ============================================
# GRADIO INTERFACE
# ============================================

def create_interface():
    with gr.Blocks(
        title="🚂 Myanmar Railway AI"
    ) as demo:

        gr.Markdown("""
        # 🚂 မြန်မာ့မီးရထား AI စက်ရုပ်

        Gemini ကို မူလ AI အဖြစ် အသုံးပြုပြီး Auto mode တွင်
        Gemini မရရှိပါက Qwen သို့ fallback လုပ်ပါမည်။
        """)

        with gr.Row():
            model_selector = gr.Dropdown(
                choices=MODEL_CHOICES,
                value="Auto (Gemini → Qwen)",
                label="AI Model",
                interactive=True,
            )

        chatbot = gr.Chatbot(
            height=400,
            label="Conversation"
        )

        with gr.Row():
            msg = gr.Textbox(
                label=(
                    "သင့်မေးခွန်းကို ရိုက်ထည့်ပါ / "
                    "Type your question"
                ),
                placeholder=(
                    "ဥပမာ - မှော်ဘီကနေ ပြည်ကို "
                    "မနက်ဖြန် ဘယ်ရထားတွေရှိလဲ"
                ),
                lines=2,
                scale=4
            )
            send_btn = gr.Button(
                "📤 Send",
                variant="primary",
                scale=1
            )

        with gr.Row():
            status_text = gr.Textbox(
                label="Status",
                value="✅ Ready",
                interactive=False,
                scale=3
            )
            clear_btn = gr.Button(
                "🗑️ Clear Chat",
                variant="secondary",
                scale=1
            )

        send_btn.click(
            fn=respond,
            inputs=[
                msg,
                chatbot,
                model_selector
            ],
            outputs=[
                msg,
                chatbot,
                status_text
            ]
        )

        msg.submit(
            fn=respond,
            inputs=[
                msg,
                chatbot,
                model_selector
            ],
            outputs=[
                msg,
                chatbot,
                status_text
            ]
        )

        clear_btn.click(
            fn=lambda: (
                [],
                "🗑️ Chat cleared"
            ),
            outputs=[
                chatbot,
                status_text
            ]
        )

        gr.api(
            json_respond,
            api_name="chat",
            api_description=(
                "RailMM chatbot JSON endpoint. "
                "Uses Auto: Gemini primary -> Qwen fallback."
            )
        )

        gr.api(
            json_respond_with_model,
            api_name="chat_with_model",
            api_description=(
                "Optional RailMM endpoint with explicit "
                "Auto/Gemini/Qwen model selection."
            )
        )

    return demo


print("🚀 Starting Railway AI (Gemini primary / Qwen fallback)...")

if "rag_engine" not in globals():
    rag_engine = RAGEngine.get_instance()

# Force LLM initialization once so provider status is visible at startup.
_ = rag_engine.llm
print("LLM status:", rag_engine.llm.get_stats())

backend_health = rag_engine.railway_api.health_check()
print("Railway backend health:", backend_health)

try:
    print(
        "Knowledge base status:",
        rag_engine.mongo.get_status()
    )
except Exception as exc:
    print(
        "⚠️ Knowledge base diagnostic failed:",
        type(exc).__name__,
        exc
    )

demo = create_interface()
demo.launch(
    share=True,
    debug=True,
    show_error=True,
    server_name="0.0.0.0",
    server_port=7860
)


🚀 Starting Railway AI (Gemini primary / Qwen fallback)...
LLM status: {'total_calls': 2, 'api_attempts': 2, 'avg_time_ms': 1463, 'total_time_ms': 2927, 'last_provider_used': 'gemini', 'last_model_used': 'gemini-3.5-flash', 'last_error': None, 'last_usage': {'cache_tokens_details': None, 'cached_content_token_count': None, 'candidates_token_count': 180, 'candidates_tokens_details': [{'modality': <MediaModality.TEXT: 'TEXT'>, 'token_count': 180}], 'prompt_token_count': 6942, 'prompt_tokens_details': [{'modality': <MediaModality.TEXT: 'TEXT'>, 'token_count': 6942}], 'thoughts_token_count': None, 'tool_use_prompt_token_count': None, 'tool_use_prompt_tokens_details': None, 'total_token_count': 7122, 'traffic_type': <TrafficType.ON_DEMAND: 'ON_DEMAND'>}, 'gemini_available': True, 'gemini_model': 'gemini-3.5-flash', 'vertex_project': 'project-067bd450-3b43-4f1f-926', 'vertex_location': 'global', 'qwen_available': True, 'qwen_model': 'qwen/qwen3.6-27b', 'default_mode': 'Auto'}
Railway backend 

🔍 Step 1: BGE-M3 direct multilingual retrieval...
📝 Original question: What can i ask u?
   Direct top score: 0.0000

📚 Retrieved Documents: 0
🌐 Translation fallback used: False

🧩 Step 3: Building multilingual prompt...
📦 Dynamic prompt size: 681 chars (stable railway knowledge is in system context)
🤖 Step 4: Generating with model mode: Auto

🔧 Native tool iteration: 1 | provider=gemini | mode=Auto

📊 LLM Stats: {'total_calls': 3, 'api_attempts': 3, 'avg_time_ms': 1560, 'total_time_ms': 4681, 'last_provider_used': 'gemini', 'last_model_used': 'gemini-3.5-flash', 'last_error': None, 'last_usage': {'cache_tokens_details': None, 'cached_content_token_count': None, 'candidates_token_count': 182, 'candidates_tokens_details': [{'modality': <MediaModality.TEXT: 'TEXT'>, 'token_count': 182}], 'prompt_token_count': 6866, 'prompt_tokens_details': [{'modality': <MediaModality.TEXT: 'TEXT'>, 'token_count': 6866}], 'thoughts_token_count': None, 'tool_use_prompt_token_count': None, 'tool_use_prompt

In [ ]:
# import torch
# import gc

# def check_memory():
#     """Check current GPU memory usage"""
#     if torch.cuda.is_available():
#         allocated = torch.cuda.memory_allocated() / 1024**3
#         reserved = torch.cuda.memory_reserved() / 1024**3
#         print(f"💾 GPU Memory:")
#         print(f"   Allocated: {allocated:.2f} GB")
#         print(f"   Reserved: {reserved:.2f} GB")
#         print(f"   Free: {(16 - reserved):.2f} GB")
#     else:
#         print("💾 CPU Memory: Not tracked")

#     # Check Python objects
#     import sys
#     print(f"📦 Python objects in memory:")
#     for obj in ['rag_engine', 'embedding_service', 'mongo', 'translator', 'llm']:
#         if obj in globals():
#             print(f"   {obj}: ✅ Loaded")
#         else:
#             print(f"   {obj}: ❌ Not loaded")

# check_memory()

In [ ]:
# import torch
# import gc
# import sys
# #
# def clear_memory():
#     """Clear GPU memory and remove loaded models"""
#     print("🧹 Cleaning memory...")

#     # List of global variables to clean
#     objects_to_clear = [
#         'rag_engine',
#         'embedding_service',
#         'mongo',
#         'translator',
#         'llm',
#         'model',
#         'tokenizer'
#     ]

#     # Clear global references
#     for obj_name in objects_to_clear:
#         if obj_name in globals():
#             obj = globals()[obj_name]
#             # Try to delete if it has a cleanup method
#             if hasattr(obj, '__del__'):
#                 try:
#                     obj.__del__()
#                 except:
#                     pass
#             # Delete from globals
#             del globals()[obj_name]
#             print(f"   Removed: {obj_name}")

#     # Clear CUDA cache
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         torch.cuda.synchronize()
#         print("   CUDA cache cleared")

#     # Run garbage collection
#     gc.collect()
#     print("   Garbage collection done")

#     # Show memory after cleanup
#     if torch.cuda.is_available():
#         allocated = torch.cuda.memory_allocated() / 1024**3
#         reserved = torch.cuda.memory_reserved() / 1024**3
#         print(f"💾 After cleanup:")
#         print(f"   Allocated: {allocated:.2f} GB")
#         print(f"   Reserved: {reserved:.2f} GB")
#         print(f"   Free: {(16 - reserved):.2f} GB")

# # Call this before re-initializing
# clear_memory()

In [ ]:
# ============================================
# RECOMMENDED PROVIDER TEST
# ============================================
#
# llm_test = HybridLLMService(
#     stable_knowledge=railway_catalog.system_knowledge
# )
#
# print(llm_test.get_stats())
#
# # Auto: Gemini first, Qwen only if Gemini fails.
# print(
#     llm_test.generate_direct(
#         "Passenger asks: ရထားအမှတ် ၇၃ ရဲ့ နာမည်ကဘာလဲ?",
#         model_mode="Auto",
#     )
# )
#
# # Force one provider:
# # model_mode="Gemini"
# # model_mode="Qwen"


In [ ]:
# embedding_service = EmbeddingService()

# mongo = MongoService()

# query = "ရန်ကုန်မှ ပဲခူးသို့ ရထား"

# embedding = embedding_service.encode(query)

# docs = mongo.vector_search(embedding)

# for d in docs:

#     print("="*60)

#     print(d["score"])

#     print(d["title"])

#     print(d["content"][:300])

In [ ]:
# ============================================================
# v7 PSEUDO-TOOL / INTERNAL-OUTPUT GUARD TEST
# ============================================================

_bad_response = """မှော်ဘီဘူတာ (station_id: 9) မှ ပြည်ဘူတာ (station_id: 7) သို့
Route ID 5 တွင် order 9 -> order 25 ဖြစ်ပါသည်။
search_schedules(from_station_id=9, to_station_id=7, route_ids=[5,16],
date_from="2026-08-22", date_to="2026-08-22")"""

_clean_response = """မနက်ဖြန် မှော်ဘီဘူတာမှ ပြည်ဘူတာသို့ သွားမည့်
ရထားအချိန်ဇယားကို စစ်ဆေးပြီး ရရှိသည့် အချိန်များကို ပြပေးပါမည်။"""

assert RAGEngine._looks_like_pseudo_tool_call(_bad_response) is True
assert RAGEngine._contains_internal_output_leak(_bad_response) is True
assert RAGEngine._looks_like_pseudo_tool_call(_clean_response) is False
assert RAGEngine._contains_internal_output_leak(_clean_response) is False

print("✅ v6 pseudo-tool and internal-output guards passed.")
